# Kaggriculture Lab - corpus ingestion at scale

Analysis workbench for the Kaggriculture competition, running where the corpus lives.

**The scale argument.** The league is published as one dataset per day, ~21.5 GB
each; the index manifest lists **56 days, 38,544 episodes, 1.19 TB**. A laptop
holds one day. Here the datasets are mounted next to the compute, so a session can
span many days with no download and no local disk pressure.

**How to ingest that much without drowning.** Full-parsing 1.19 TB of replays is
the wrong first move. This notebook works in tiers:

| tier | what it reads | cost | answers |
| --- | --- | --- | --- |
| 0 | every `manifest.csv` | kilobytes | counts, score bands, sizes per day |
| 1 | the first ~8 kB of every replay | ~300 MB total | team names, seeds, per-team history |
| 2 | full replay JSON | ~35 MB per episode | behaviour: roster, market, revenue, timing |

**On the GPU, honestly:** parsing JSON is CPU- and I/O-bound, and Kaggle's own
docs say pandas/sklearn do not benefit from a GPU. Worse, GPU sessions have
historically come with *fewer* cores and *less* RAM than CPU ones, which is the
opposite of what a bulk-ingest job wants. Cell 1 prints the resources the session
actually has, so the choice is made from evidence instead of a habit. The GPU earns
its keep only for a genuinely numeric job (a market surrogate trained on the
trajectories), which is a separate project.

**Which conclusions are safe.** The corpus is published as "top episode replays"
and its own manifest shows the scores clustered around 3000, while our submissions
sit near 1200: it is a different rating band, so it supports WITHIN-GAME
comparisons (both seats of one match) and engine/market mechanics, but not
comparisons of our play against theirs.

In [ ]:
import collections
import csv
import glob
import json
import os
import re
import sys
import time

print("python", sys.version.split()[0])
print(f"cpu_count          : {os.cpu_count()}")
try:
    with open("/proc/meminfo") as handle:
        for line in handle:
            if line.startswith(("MemTotal", "MemAvailable")):
                print(f"{line.split(':')[0]:<19}: {int(line.split()[1]) / 1e6:.1f} GB")
except OSError:
    pass
for cmd in ("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",):
    try:
        import subprocess
        print("gpu                :", subprocess.run(cmd.split(), capture_output=True,
              text=True, timeout=20).stdout.strip() or "none")
    except Exception as exc:
        print("gpu                : unavailable", type(exc).__name__)

# Discover the layout instead of assuming it: the first version of this notebook
# globbed /kaggle/input/<slug>/ and silently found nothing.
def find_all(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))


manifests = find_all("manifest.csv")
episode_files = find_all("*.json")
print(f"\nmanifests found   : {len(manifests)}")
print(f"episode files     : {len(episode_files):,}")
for path in manifests[:6]:
    print(f"  {path}  ({os.path.getsize(path) / 1e3:.0f} kB)")

## Tier 0 - the manifests (kilobytes, whole league)

Each daily dataset ships `manifest.csv` with per-episode metadata
(`episode_id, create_time, avg_score, min_score, sum_score, agent_count,
size_bytes`), and the index dataset lists the daily datasets. This answers the
population questions before touching a replay.

In [ ]:
episodes = []      # per-episode rows, from every daily dataset's manifest
index_rows = []    # the index dataset's list of daily datasets

for manifest in manifests:
    source = os.path.basename(os.path.dirname(manifest))
    try:
        with open(manifest) as handle:
            for row in csv.DictReader(handle):
                if "episode_id" in row:
                    row["_source"] = source
                    episodes.append(row)
                elif "daily_dataset_slug" in row:
                    index_rows.append(row)
    except OSError as exc:
        print("skip", manifest, exc)

print(f"episodes with metadata : {len(episodes):,} across "
      f"{len({e['_source'] for e in episodes})} attached days")
if index_rows:
    print(f"league per the index   : {sum(int(r['episode_count']) for r in index_rows):,} "
          f"episodes, {sum(int(r['total_bytes']) for r in index_rows) / 1e12:.2f} TB, "
          f"{len(index_rows)} days")
    print(f"newest day listed      : {max(r['date'] for r in index_rows)}")

by_day = collections.defaultdict(list)
for row in episodes:
    if row.get("avg_score"):
        by_day[row["_source"]].append(float(row["avg_score"]))
if by_day:
    print(f"\n{'day dataset':<42}{'n':>6}{'min':>7}{'median':>8}{'max':>7}")
    for day in sorted(by_day)[:12]:
        scores = sorted(by_day[day])
        print(f"{day:<42}{len(scores):>6}{scores[0]:>7.0f}"
              f"{scores[len(scores) // 2]:>8.0f}{scores[-1]:>7.0f}")

## The daily/event-time extractor (single source, embedded)

This cell is `scripts/corpus_economic_day.py` verbatim (minus its CLI), so the
extractor the notebook runs is the one the workstation tests exercise. It pins
the replay action offset (`action[i]` produced `observation[i]`), takes the
declared treatment as the OBSERVED 3-to-4 `unlocked_quadrants` transition, and
refuses to report requested `SELL` units as executed sales.

In [ ]:
#!/usr/bin/env python
"""Day-by-day economic extraction for ONE Kaggle daily episode dataset.

Why this module exists
----------------------
The one-day economic event study needs a daily/event-time extractor that can be
unit-tested on the workstation but runs where the daily replay datasets are
mounted. Kaggle is the only place those datasets live, so the notebook builder
embeds this file verbatim as a cell and the tests import it directly: one
implementation of the guard and the extractor, two call sites.

Two hard invariants
-------------------
* EXACTLY ONE daily episode dataset per run. Zero or more than one means the
  run is misconfigured and must fail closed *before* any replay is parsed; the
  index dataset, engine tables and reference agents are not daily sources.
* The treatment timestamp is the OBSERVED ``farm.unlocked_quadrants`` 3 -> 4
  transition, not the ``BUY_LAND`` request, which can silently fail.

Replay semantics pinned here (established in
``kaggriculture.agents.replay_opponent``; ``ACTION_OFFSET == 1``)
------------------------------------------------------------------------
* ``steps[i][seat]["observation"]`` is the state an agent sees at turn ``i``.
* ``steps[i][seat]["action"]`` PRODUCED ``observation[i]``: it acted on the
  state at index ``i - 1``. So the ``BUY_LAND`` request that turns three
  quadrants into four sits at index ``i`` when ``observation[i]`` is the first
  to show four. Index 0 is a placeholder action and is never a request.
* A daily snapshot is the LAST observation of an in-game day, so the $4,000
  purchase that lands within the day is visible in that day's settled cash.

What is deliberately NOT measured
---------------------------------
A ``SELL`` order stores the units an agent *requested*, not the units the market
executed, and the replay does not expose the fills. Sold-unit accounting would
therefore be fabricated, so this module omits it entirely. Only ``farm.money``
is called cash; net cash movement is never called revenue or income.
"""

from __future__ import annotations

import argparse
import collections
import glob
import os
import re
import statistics

#: Engine turns per in-game day, used only when an observation lacks ``day``.
TURNS_PER_DAY = 24

#: Event-time window relative to the buyer's observed 3 -> 4 unlock day.
EVENT_LO, EVENT_HI = -4, 8

#: A mounted daily episode dataset is a directory whose name matches this.
DAILY_DATASET_RE = re.compile(r"^kaggriculture-episodes-(\d{4}-\d{2}-\d{2})$")

#: Boundary between embeddable library code and the local CLI. The notebook
#: builder cuts the source at this line so the cell never defines ``main`` and
#: never runs the CLI when a notebook executes with ``__name__ == "__main__"``.
CLI_MARKER = "# --- command-line entry point (not embedded in the notebook) ---"


class OneDayGuardError(RuntimeError):
    """Raised when a run is not bound to exactly one daily episode dataset."""


class MatchSkip(Exception):
    """A replay that cannot supply a bounded primary pair; its reason is kept."""


# --------------------------------------------------------------------------
# One-day guard
# --------------------------------------------------------------------------

def daily_dataset_date(name):
    """Return the ``YYYY-MM-DD`` of a daily dataset path/name, or ``None``.

    Only the basename is inspected, so both the flat Kaggle mount
    (``/kaggle/input/kaggriculture-episodes-2026-09-24``) and the nested one
    (``/kaggle/input/datasets/kaggle/kaggriculture-episodes-2026-09-24``) map to
    the same date. Index, engine and reference-agent names do not match.
    """
    base = os.path.basename(os.path.normpath(str(name)))
    match = DAILY_DATASET_RE.match(base)
    return match.group(1) if match else None


def daily_dataset_dates(names):
    """Map ``date -> sorted distinct matching source names``."""
    found = {}
    for name in names:
        date = daily_dataset_date(name)
        if date is not None:
            found.setdefault(date, set()).add(str(name))
    return {date: sorted(values) for date, values in found.items()}


def daily_dataset_sources(names):
    """Sorted matching daily source names, duplicates preserved.

    The guard counts RAW matches, not distinct dates: two mounts of the same day
    (or a duplicated ``dataset_sources`` entry) must fail closed, because the run
    would otherwise read the same day twice and call it one.
    """
    return sorted(str(name) for name in names if daily_dataset_date(name) is not None)


def require_exactly_one_day(names):
    """Fail closed unless exactly one daily episode dataset mount is present.

    Returns ``(date, source)``. Raises :class:`OneDayGuardError` for zero daily
    sources, for two or more distinct dates, and for two matching mounts of the
    SAME date. Non-daily sources are ignored on purpose, not treated as error.
    """
    sources = daily_dataset_sources(names)
    if not sources:
        raise OneDayGuardError(
            "no daily episode dataset mounted (expected kaggriculture-episodes-"
            "YYYY-MM-DD); refusing to guess")
    if len(sources) > 1:
        raise OneDayGuardError(
            "more than one daily episode dataset mounted: "
            + ", ".join(sources)
            + "; one run must analyse exactly one mount, even when the duplicate "
            "paths share a date")
    return daily_dataset_date(sources[0]), sources[0]


def discover_daily_dirs(root="/kaggle/input"):
    """Directories under ``root`` whose name looks like a daily dataset.

    Discovery only reads names: it never opens a replay, so it is safe to call
    before the full-parse pass.
    """
    seen = set()
    pattern = os.path.join(root, "**", "kaggriculture-episodes-*")
    for path in glob.glob(pattern, recursive=True):
        if os.path.isdir(path):
            seen.add(path)
    return sorted(seen)


def require_expected_day(discovered, expected):
    """Fail closed when the full-parse pass disagrees with the one-day guard.

    The Tier-2 pass re-derives the day from replay paths; if that set is not the
    single day the guard resolved, the run would analyse a different sample than
    the report claims, so it stops instead of continuing.
    """
    if list(discovered) != [expected]:
        raise OneDayGuardError(
            "tier-2 discovered days " + repr(list(discovered))
            + " but the one-day guard resolved " + repr(expected))
    return expected


def require_within_cap(total, cap, expected):
    """Fail closed rather than truncate the mounted one-day file list.

    With one day mounted, every replay in it is part of the sample; silently
    slicing at ``cap`` would shrink the day while the report still named it.
    """
    if total > cap:
        raise OneDayGuardError(
            f"{total:,} replays mounted for {expected} exceed the cap {cap:,}; "
            "raise the cap instead of truncating a single day")
    return total


# --------------------------------------------------------------------------
# Observation readers (observed state only)
# --------------------------------------------------------------------------

def _seat_state(steps, index, seat):
    if index < 0 or index >= len(steps):
        return None
    row = steps[index]
    if not isinstance(row, (list, tuple)) or seat >= len(row):
        return None
    state = row[seat]
    return state if isinstance(state, dict) else None


def _observation(steps, index, seat):
    state = _seat_state(steps, index, seat)
    if state is None:
        return None
    obs = state.get("observation")
    return obs if isinstance(obs, dict) else None


def _farm(steps, index, seat):
    """The farm of ``seat`` at ``index`` (indexed by ``observation["player"]``)."""
    obs = _observation(steps, index, seat)
    if obs is None:
        return {}
    farms = obs.get("farms")
    if not isinstance(farms, (list, tuple)):
        return {}
    player = obs.get("player")
    if not isinstance(player, int) or not (0 <= player < len(farms)):
        player = seat
    farm = farms[player] if 0 <= player < len(farms) else None
    return farm if isinstance(farm, dict) else {}


def _number(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return 0.0
    return float(value)


def _sum_counts(value):
    """Total units across a ``{product: units}`` dict or a list of such dicts."""
    if isinstance(value, dict):
        return sum(int(v) for v in value.values()
                   if isinstance(v, (int, float)) and not isinstance(v, bool))
    if isinstance(value, (list, tuple)):
        return sum(_sum_counts(item) for item in value)
    return 0


def _tile_counts(farm):
    """(planted cells, animal cells) over the farm's tile grid."""
    crops = animals = 0
    tiles = farm.get("tiles")
    if not isinstance(tiles, (list, tuple)):
        return crops, animals
    for row in tiles:
        if not isinstance(row, (list, tuple)):
            continue
        for cell in row:
            if not isinstance(cell, dict):
                continue
            if cell.get("animal"):
                animals += 1
            elif cell.get("kind") == "PLANT":
                crops += 1
    return crops, animals


def _quadrants(farm):
    unlocked = farm.get("unlocked_quadrants")
    return len(unlocked) if isinstance(unlocked, (list, tuple)) else 0


def _day_of(obs, index):
    if isinstance(obs, dict):
        day = obs.get("day")
        if isinstance(day, (int, float)) and not isinstance(day, bool):
            return int(day)
    return index // TURNS_PER_DAY


# --------------------------------------------------------------------------
# Observed snapshots, unlock and pair classification
# --------------------------------------------------------------------------

def observed_snapshot(steps, index, seat):
    """Observable economy of ``seat`` at one turn.

    No requested or executed market data enters here: ``SELL`` quantities are
    fills the replay does not expose, so they are not represented at all.
    """
    obs = _observation(steps, index, seat)
    farm = _farm(steps, index, seat)
    private = obs.get("private") if isinstance(obs, dict) else None
    if not isinstance(private, dict):
        private = {}
    crops, animals = _tile_counts(farm)
    hands = farm.get("hands")
    return {
        "day": _day_of(obs, index),
        "cash": _number(farm.get("money")),
        "quadrants": _quadrants(farm),
        "crop_tiles": crops,
        "animals": animals,
        "hands": len(hands) if isinstance(hands, (list, tuple)) else 0,
        "shed_units": _sum_counts(private.get("shed")),
        "carry_units": _sum_counts(private.get("inventories")),
    }


def daily_snapshots(steps, seat):
    """One settled row per in-game day: the LAST observation of that day.

    The day's final turn already contains any purchase made during the day, so
    a single observation is the honest daily snapshot. Change from the prior day
    is attached (``cash_change``), never averaged or interpolated.
    """
    last_index = {}
    for index in range(len(steps)):
        obs = _observation(steps, index, seat)
        if obs is None:
            continue
        last_index[_day_of(obs, index)] = index
    rows = []
    previous_cash = None
    for day in sorted(last_index):
        row = observed_snapshot(steps, last_index[day], seat)
        row["day"] = day
        row["cash_change"] = None if previous_cash is None else row["cash"] - previous_cash
        previous_cash = row["cash"]
        rows.append(row)
    return rows


def buy_land_indices(steps, seat):
    """Indices ``i`` where ``action[i]`` requests ``BUY_LAND``.

    Because ``action[i]`` produced ``observation[i]``, these are exactly the
    transitions ``observation[i-1] -> observation[i]`` the request could affect.
    Index 0 is the placeholder action and is never a request.
    """
    found = []
    for index in range(len(steps)):
        state = _seat_state(steps, index, seat)
        action = (state or {}).get("action")
        if not isinstance(action, dict):
            continue
        for order in action.get("market") or []:
            if isinstance(order, (list, tuple)) and order and order[0] == "BUY_LAND":
                found.append(index)
                break
    return found


def observed_unlock(steps, seat, target=4):
    """First observed ``(day, index)`` where the farm shows ``target`` quadrants.

    Returns ``(None, None)`` when the state never reaches ``target``. A
    ``BUY_LAND`` request with no such observed transition is deliberately not a
    treatment: the request is allowed to fail silently.
    """
    for index in range(len(steps)):
        if _quadrants(_farm(steps, index, seat)) >= target:
            return _day_of(_observation(steps, index, seat), index), index
    return None, None


def final_quadrants(steps, seat):
    """Last observed quadrant count for a seat (0 when no observation exists)."""
    last = 0
    for index in range(len(steps)):
        count = _quadrants(_farm(steps, index, seat))
        if count:
            last = count
    return last


def classify_pair(steps):
    """Classify one replay by its two final observed quadrant counts.

    The primary cohort is exactly one seat at four quadrants and its opponent at
    three. Everything else is a named, separate configuration.
    """
    quads = [final_quadrants(steps, 0), final_quadrants(steps, 1)]
    counts = sorted(quads)
    if counts == [3, 4]:
        label = "primary_one4_one3"
    elif counts == [4, 4]:
        label = "both4"
    elif counts == [3, 3]:
        label = "both3"
    elif counts[-1] == 4:
        label = "one4_other_not3"
    elif counts[-1] < 3:
        label = "both_under3"
    else:
        label = "neither_reached4"
    return label, quads


def extract_match(steps, teams=None):
    """Bounded per-match extraction, both seats of the SAME replay.

    Returns a dict with the pair class and, for the primary cohort only, the
    buyer's and opponent's daily rows, the observed unlock day and the matching
    ``BUY_LAND`` request. Non-primary matches return without any snapshot work,
    so the fold stays cheap.
    """
    if not isinstance(steps, (list, tuple)) or len(steps) < 2:
        raise MatchSkip("truncated: fewer than two steps")
    label, quads = classify_pair(steps)
    result = {
        "class": label,
        "quadrants": quads,
        "teams": [str(team) for team in (teams or [])],
        "buyer": None,
        "opponent": None,
    }
    if label != "primary_one4_one3":
        return result

    buyer = 0 if quads[0] == 4 else 1
    opponent = 1 - buyer
    buyer_rows = daily_snapshots(steps, buyer)
    opponent_rows = daily_snapshots(steps, opponent)
    if not buyer_rows or not opponent_rows:
        raise MatchSkip("primary pair without daily snapshots")
    unlock_day, unlock_index = observed_unlock(steps, buyer, 4)
    if unlock_day is None:
        raise MatchSkip("four quadrants at the end but no observed 3 -> 4 transition")

    requests = buy_land_indices(steps, buyer)
    prior = [index for index in requests if index <= unlock_index]
    request_gap = None if not prior else unlock_index - max(prior)
    result["buyer"] = {
        "seat": buyer,
        "rows": buyer_rows,
        "unlock_day": unlock_day,
        "unlock_index": unlock_index,
        "buy_land_indices": requests,
        "request_gap": request_gap,
    }
    result["opponent"] = {"seat": opponent, "rows": opponent_rows}
    return result


def pair_rows(match):
    """Day-aligned buyer-versus-opponent rows for one primary match."""
    buyer = match["buyer"]
    opponent = match["opponent"]
    opponent_by_day = {row["day"]: row for row in opponent["rows"]}
    rows = []
    for brow in buyer["rows"]:
        orow = opponent_by_day.get(brow["day"])
        if orow is None:
            continue
        rows.append({
            "day": brow["day"],
            "buyer_cash": brow["cash"],
            "opp_cash": orow["cash"],
            "gap": brow["cash"] - orow["cash"],
            "buyer_change": brow["cash_change"],
            "opp_change": orow["cash_change"],
            "buyer_quadrants": brow["quadrants"],
            "opp_quadrants": orow["quadrants"],
            "buyer_crop": brow["crop_tiles"],
            "opp_crop": orow["crop_tiles"],
            "buyer_animal": brow["animals"],
            "opp_animal": orow["animals"],
            "buyer_hands": brow["hands"],
            "opp_hands": orow["hands"],
            "buyer_shed": brow["shed_units"],
            "opp_shed": orow["shed_units"],
            "buyer_carry": brow["carry_units"],
            "opp_carry": orow["carry_units"],
        })
    return rows


# --------------------------------------------------------------------------
# Bounded aggregation
# --------------------------------------------------------------------------

def new_summary():
    """Accumulator for the primary cohort across one full-parse pass."""
    return {
        "parsed": 0,
        "primary": 0,
        "buyer_wins": 0,
        "classes": collections.Counter(),
        "skipped": collections.Counter(),
        "primary_teams": set(),
        "days": collections.defaultdict(list),
        "events": collections.defaultdict(list),
        "unlock_days": [],
        "final_gap": [],
        "request_ok": 0,
        "request_missing": 0,
    }


def fold_match(summary, match):
    """Fold one :func:`extract_match` result into ``summary``.

    Only the primary cohort feeds the daily/event tables; every class is counted
    so the report can say how many replays were excluded and why.
    """
    summary["parsed"] += 1
    summary["classes"][match["class"]] += 1
    if match["class"] != "primary_one4_one3" or match["buyer"] is None:
        return

    rows = pair_rows(match)
    if not rows:
        summary["skipped"]["primary pair lost day alignment"] += 1
        return

    summary["primary"] += 1
    # Only the primary cohort's teams belong in the headline: teams seen in the
    # excluded classes must not inflate the distinct-team denominator.
    for team in match.get("teams") or ():
        summary["primary_teams"].add(team)
    buyer = match["buyer"]
    unlock_day = buyer["unlock_day"]
    summary["unlock_days"].append(unlock_day)
    if buyer["request_gap"] is not None and buyer["request_gap"] <= 1:
        summary["request_ok"] += 1
    else:
        summary["request_missing"] += 1

    summary["final_gap"].append(rows[-1]["gap"])
    if rows[-1]["gap"] > 0:
        summary["buyer_wins"] += 1
    for row in rows:
        summary["days"][row["day"]].append(row)
        offset = row["day"] - unlock_day
        if EVENT_LO <= offset <= EVENT_HI:
            summary["events"][offset].append(row)


def _median(values):
    return statistics.median(values) if values else None


def _median_of(rows, key):
    return _median([row[key] for row in rows if row.get(key) is not None])


def _money(value):
    return "-" if value is None else f"{value:,.0f}"


def _signed(value):
    return "-" if value is None else f"{value:+,.0f}"


def _count(value):
    return "-" if value is None else f"{value:.0f}"


def _render_table(lines, headers, pairs):
    """Append a fixed-width median table; ``pairs`` is (label, cells) per row."""
    lines.append("".join(f"{head:>{width}}" for head, width in headers))
    for label, cells in pairs:
        body = "".join(f"{text:>{headers[index + 1][1]}}"
                       for index, text in enumerate(cells))
        lines.append(f"{label:>4}" + body)


def render_report(summary, date):
    """Return the bounded (< 20 kB) text report for one dataset date.

    Wording is deliberate: every column is cash or a count, denominators are
    printed, and the closing note states the analysis is descriptive.
    """
    lines = []
    add = lines.append
    add("=" * 100)
    add(f"DAILY ECONOMIC EVENT STUDY - dataset date {date} (one day per run)")
    add("=" * 100)
    add(f"replays parsed        : {summary['parsed']:,}")
    add(f"primary matches       : {summary['primary']:,} "
        "(one seat 4 quadrants, the other 3, same replay)")
    add(f"distinct teams         : {len(summary['primary_teams']):,} "
        "(primary cohort only; shared across replays, rows not independent people)")
    if summary["primary"]:
        share = summary["buyer_wins"] / summary["primary"]
        add(f"buyer win share        : {summary['buyer_wins']:,}/"
            f"{summary['primary']:,} = {share:.1%} "
            f" (denominator: {summary['primary']:,} primary matches)")
    else:
        add("buyer win share        : n/a (no primary match)")
    add("")
    add("classes of the two seats (final observed quadrants, same replay):")
    for label, count in sorted(summary["classes"].items(), key=lambda kv: -kv[1]):
        add(f"  {label:<22} {count:>7,}")
    if summary["skipped"]:
        add(f"skipped replays: {sum(summary['skipped'].values()):,}")
        for reason, count in summary["skipped"].most_common():
            add(f"  {count:>7,}  {reason}")
    else:
        add("skipped replays: 0")

    if not summary["primary"]:
        add("")
        add("No primary match on this day; nothing to table. The class counts above")
        add("are the honest result and no causal reading is possible from an empty cohort.")
        return "\n".join(lines)

    unlock_days = summary["unlock_days"]
    add("")
    add(f"observed unlock day (3 -> 4 quadrants): median {_median(unlock_days):.0f}"
        f" (min {min(unlock_days):.0f}, max {max(unlock_days):.0f},"
        f" n={len(unlock_days):,})")
    total_req = summary["request_ok"] + summary["request_missing"]
    add("  of those, a BUY_LAND request sits at or one step before the observed")
    add(f"  transition: {summary['request_ok']:,}/{total_req:,} = "
        f"{summary['request_ok'] / total_req:.0%}. The request alone is never the")
    add("  treatment; a silent BUY_LAND failure leaves the request but no 4-quadrant")
    add("  transition, so only the observed state counts.")

    cash_headers = [("day", 4), ("n", 6), ("buyer_cash", 12), ("opp_cash", 12),
                    ("gap", 10), ("buyer_chg", 11), ("opp_chg", 10)]
    resource_headers = [("day", 4), ("n", 6), ("b_q", 4), ("o_q", 4),
                        ("b_crop", 7), ("o_crop", 7), ("b_an", 5), ("o_an", 5),
                        ("b_hand", 7), ("o_hand", 7), ("b_shed", 7), ("o_shed", 7),
                        ("b_carry", 8), ("o_carry", 8)]

    def cash_cells(rows):
        return [
            str(len(rows)),
            _money(_median_of(rows, "buyer_cash")),
            _money(_median_of(rows, "opp_cash")),
            _signed(_median_of(rows, "gap")),
            _signed(_median_of(rows, "buyer_change")),
            _signed(_median_of(rows, "opp_change")),
        ]

    def resource_cells(rows):
        return [
            str(len(rows)),
            _count(_median_of(rows, "buyer_quadrants")),
            _count(_median_of(rows, "opp_quadrants")),
            _count(_median_of(rows, "buyer_crop")),
            _count(_median_of(rows, "opp_crop")),
            _count(_median_of(rows, "buyer_animal")),
            _count(_median_of(rows, "opp_animal")),
            _count(_median_of(rows, "buyer_hands")),
            _count(_median_of(rows, "opp_hands")),
            _count(_median_of(rows, "buyer_shed")),
            _count(_median_of(rows, "opp_shed")),
            _count(_median_of(rows, "buyer_carry")),
            _count(_median_of(rows, "opp_carry")),
        ]

    add("")
    add("--- IN-GAME DAY: cash snapshots (median; day = LAST observed turn) ---")
    add("  cash is farm.money. buyer and opponent are both seats of the same replay.")
    _render_table(lines, cash_headers,
                  [(str(day), cash_cells(summary["days"][day]))
                   for day in sorted(summary["days"])])

    add("")
    add("--- IN-GAME DAY: observable resources (median counts) ---")
    _render_table(lines, resource_headers,
                  [(str(day), resource_cells(summary["days"][day]))
                   for day in sorted(summary["days"])])

    add("")
    add("--- EVENT TIME: days relative to the buyer's observed unlock (median) ---")
    _render_table(lines, cash_headers,
                  [(f"{offset:+d}", cash_cells(summary["events"][offset]))
                   for offset in sorted(summary["events"])])

    add("")
    add("--- EVENT TIME: observable resources (median counts) ---")
    _render_table(lines, resource_headers,
                  [(f"{offset:+d}", resource_cells(summary["events"][offset]))
                   for offset in sorted(summary["events"])])

    pretrend_values = [row["gap"] for offset in range(EVENT_LO, 0)
                       for row in summary["events"].get(offset, [])]
    pretrend = _median(pretrend_values)
    dip_rows = summary["events"].get(0, [])
    dip = _median_of(dip_rows, "buyer_change")
    dip_mechanical = sum(1 for row in dip_rows
                         if row["buyer_change"] is not None and row["buyer_change"] <= -3500)
    gap_by_offset = {offset: _median_of(rows, "gap")
                     for offset, rows in summary["events"].items()}
    recovery = None
    for offset in range(1, EVENT_HI + 1):
        value = gap_by_offset.get(offset)
        if pretrend is not None and value is not None and value >= pretrend:
            recovery = offset
            break
    crossover = None
    for offset in range(EVENT_HI + 1):
        value = gap_by_offset.get(offset)
        if value is not None and value >= 0:
            crossover = offset
            break

    add("")
    add("--- HEADLINE (descriptive, not causal) ---")
    add(f"pretrend median gap, offsets {EVENT_LO}..-1 : {_signed(pretrend)}"
        + (f"  (n={len(pretrend_values):,} day-rows)" if pretrend_values else "  (no rows)"))
    add(f"purchase-day (offset 0) buyer cash change  : {_signed(dip)}"
        f"  median over n={len(dip_rows):,}; includes the mechanical -$4,000 land purchase")
    if dip_rows:
        add(f"  purchase-day rows at or below -$3,500    : {dip_mechanical:,}/"
            f"{len(dip_rows):,} = {dip_mechanical / len(dip_rows):.0%}")
    add("recovery (first offset > 0 with median gap >= pretrend): "
        + ("not within +" + str(EVENT_HI) if recovery is None else f"offset +{recovery}"))
    add("crossover (first offset >= 0 with median gap >= 0)     : "
        + ("not within +" + str(EVENT_HI) if crossover is None else f"offset +{crossover}"))
    add("final median gap (last day of each match)             : "
        f"{_signed(_median(summary['final_gap']))}")
    add("")
    add("Denominators: every table row prints its own n; primary matches above are")
    add("the only rows in the day and event tables. Cash is farm.money, not revenue;")
    add("requested SELL units are not executed fills and are omitted on purpose.")
    add("A within-game control removes shared shops and market, not policy, skill or")
    add("selection; repeated teams make rows non-independent. One date per run.")
    return "\n".join(lines)


## The shop-window demand extractor (single source, embedded)

This cell is `scripts/corpus_shop_windows.py` verbatim (minus its CLI), so the
shop analysis the notebook runs is the one the workstation tests exercise. It
reads the per-turn `town.unlocked_shops` snapshots (duplicates are independent
consumers), prices expected demand by joining the engine's shop-product mapping
with the mounted `town_shop_demand.csv` (single-product shops consume double, the
town centre is excluded), and measures EXECUTED units as holdings drops on a
`SELL` turn - never as the requested quantity.

In [ ]:
#!/usr/bin/env python
"""Shop-window demand analysis for ONE Kaggle daily episode dataset.

Why this module exists
----------------------
The town shop system is PUBLIC information in every observation
(``obs.town.unlocked_shops``): a shop unlocks every 3 in-game days (with
replacement, capped at 8 live instances) and each instance consumes on its own
4-turn schedule. Before shop-awareness becomes a first-class parameter of the
policy agent, the corpus has to show whether selling into the shop calendar
actually separates winners from losers.

This module extracts that evidence in the SAME Tier-2 full-parse pass as the
daily economic study: one read of the replay feeds both reports. It is stdlib
only and embeddable exactly like ``scripts/corpus_economic_day.py`` - the
notebook builder cuts the source at :data:`SHOP_CLI_MARKER`, so the cell never
defines ``main`` and a notebook running with ``__name__ == "__main__"`` never
executes a CLI.

Namespace rule (why every top-level name is prefixed)
-----------------------------------------------------
The notebook embeds this module AND ``scripts/corpus_economic_day.py`` into the
SAME kernel namespace, the economic cell first, and the other cells keep writing
into that same namespace afterwards. A top-level name is therefore never a
private detail. Two crashes came from that, one per direction:

* v11: the shop module defined ``render_report``, which the later cell silently
  replaced, so the economic report was served the shop renderer's summary and
  raised ``KeyError: 'demand_table'``.
* v12: the Tier-2 loop header ``for _cut, (_day, _path) in enumerate(sample, 1)``
  left ``_day`` bound to the day STRING, overwriting this module's ``_day``
  formatter, so ``shop_render_report`` died calling a ``str``.

So EVERY top-level name this module defines carries a ``shop_``/``SHOP_``(or
``_shop_``) prefix, not only the names it happens to share with the economic
module today: a short helper like ``_day`` or ``holdings`` can be re-bound by any
cell that comes later. ``tests/test_lab_one_day.py`` pins both halves of that:
``test_shop_top_level_names_all_carry_the_shop_prefix`` rejects a new un-prefixed
name, ``test_embedded_extractors_share_no_top_level_name`` rejects a collision
with the economic cell, and
``test_renderers_survive_the_shared_notebook_namespace`` executes both cells with
every other cell's top-level bindings in place and calls both renderers, which is
the regression the v12 crash taught.

Engine semantics pinned here (``kaggriculture`` source + ``docs/engine-rule-sheet.md``)
------------------------------------------------------------------------------------
* ``SHOPS`` maps each shop name to the products it consumes.
* Every ``townShopSellInterval`` = 4 turns, each live instance consumes 1 unit of
  each of its products, DOUBLED (2 units) when the shop has a single product.
  So per instance per product per day: ``24 / 4 * (2 if single else 1)`` = 12 or 6.
* ``townShopUnlockInterval`` = 3 days; ``SHOP_MAX_INSTANCES`` = 8; draws are with
  replacement, so duplicates are legitimate and each copy consumes independently.
* The town centre drains 1 unit of every non-fertilizer product per 24 turns and
  is EXCLUDED here: this module measures SHOP demand only.

Replay semantics pinned here (established in ``kaggriculture.agents.replay_opponent``;
``ACTION_OFFSET == 1``)
------------------------------------------------------------------------------
* ``steps[i][seat]["action"]`` PRODUCED ``observation[i]``; it acted on state
  ``i - 1``. Index 0 is a placeholder action and is never a request.
* Executed sells are holdings DROPS (``private.shed`` + ``private.inventories``)
  on a turn that requests ``SELL`` for that product. The requested quantity is
  NOT a fill: the strong tape submits ``SELL <item> 10000`` as "sell everything",
  so reporting order quantities as sales overstates them by orders of magnitude.
  A requested-but-unfilled ``SELL`` leaves holdings unchanged and therefore
  counts zero units.
"""

from __future__ import annotations

import argparse
import collections
import csv
import glob
import os
import statistics

#: Engine turns per in-game day.
SHOP_TURNS_PER_DAY = 24

#: ``townShopSellInterval``: every instance consumes on this turn cadence.
SHOP_CONSUME_INTERVAL_TURNS = 4

#: ``townShopUnlockInterval``: one new (possibly duplicate) instance every N days.
SHOP_UNLOCK_INTERVAL_DAYS = 3

#: A shop whose product list has a single entry consumes double per tick.
SHOP_SINGLE_PRODUCT_MULTIPLIER = 2

#: ``SHOP_MAX_INSTANCES``: hard cap on live instances, not on variety.
SHOP_MAX_INSTANCES = 8

#: ``SHOPS`` from the engine, verbatim: which products each shop consumes.
SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}

#: Products whose demand any shop can draw (union of :data:`SHOP_PRODUCTS`).
SHOP_CONSUMING_PRODUCTS = tuple(sorted(
    {product for products in SHOP_PRODUCTS.values() for product in products}))

#: Products no shop consumes: MELON and FERTILIZER are town-centre / market only.
SHOP_NON_PRODUCTS = ("MELON", "FERTILIZER")

#: The tape's one observed shop gate (route switch at step 226): YARN_STORE/wool.
SHOP_YARN_SHOP = "YARN_STORE"
SHOP_YARN_PRODUCT = "WOOL"

#: Mounted engine table that pins the per-product per-instance rates.
SHOP_DEMAND_FILENAME = "town_shop_demand.csv"

#: Local cache of the engine table (also discovered recursively under /kaggle/input).
SHOP_LOCAL_DEMAND_PATHS = ("data/engine-tables/town_shop_demand.csv",)

#: Boundary between embeddable library code and the local CLI. The notebook
#: builder cuts the source at this line so the cell never defines ``main``.
SHOP_CLI_MARKER = "# --- command-line entry point (not embedded in the notebook) ---"


class ShopSkip(Exception):
    """A replay that cannot supply a shop-window extraction; its reason is kept."""


# --------------------------------------------------------------------------
# Observation readers (observed state only)
# --------------------------------------------------------------------------

def _shop_state(steps, index, seat):
    if index < 0 or index >= len(steps):
        return None
    row = steps[index]
    if not isinstance(row, (list, tuple)) or seat >= len(row):
        return None
    state = row[seat]
    return state if isinstance(state, dict) else None


def _shop_observation(steps, index, seat):
    state = _shop_state(steps, index, seat)
    if state is None:
        return None
    obs = state.get("observation")
    return obs if isinstance(obs, dict) else None


def _shop_day_of(obs, index):
    if isinstance(obs, dict):
        day = obs.get("day")
        if isinstance(day, (int, float)) and not isinstance(day, bool):
            return int(day)
    return index // SHOP_TURNS_PER_DAY


def shop_observed_shops(steps, index, seat):
    """The ``town.unlocked_shops`` list at one turn, or ``None`` if unreadable.

    The list is returned as-is: duplicates are preserved because each copy is an
    independent consumer (the draw is with replacement).
    """
    obs = _shop_observation(steps, index, seat)
    if obs is None:
        return None
    town = obs.get("town")
    if not isinstance(town, dict):
        return None
    shops = town.get("unlocked_shops")
    if not isinstance(shops, (list, tuple)):
        return None
    return [str(shop) for shop in shops]


def shop_observed_last_day(steps, seat):
    """Last in-game day seen for ``seat``, or ``None`` when there is no state."""
    last = None
    for index in range(len(steps)):
        obs = _shop_observation(steps, index, seat)
        if obs is None:
            continue
        last = _shop_day_of(obs, index)
    return last


def shop_final_money(steps, seat):
    """Last observed ``farm.money`` for ``seat`` (0.0 when no observation)."""
    money = 0.0
    for index in range(len(steps)):
        obs = _shop_observation(steps, index, seat)
        if obs is None:
            continue
        farms = obs.get("farms")
        if not isinstance(farms, (list, tuple)):
            continue
        player = obs.get("player")
        if not isinstance(player, int) or not (0 <= player < len(farms)):
            player = seat
        farm = farms[player] if 0 <= player < len(farms) else None
        if isinstance(farm, dict):
            value = farm.get("money")
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                money = float(value)
    return money


# --------------------------------------------------------------------------
# Shop timeline
# --------------------------------------------------------------------------

def shop_timeline(steps, seat):
    """Observed per-instance shop unlock timeline for one seat.

    Iterates the ``town.unlocked_shops`` snapshots turn by turn and emits one
    entry per NEW observed instance: ``{"shop", "turn", "day"}``. Duplicates are
    preserved as separate entries with their own unlock turn, because two BAKERY
    instances unlock at different times and consume independently. A shop is
    never removed by the engine, so the instance count is monotone and a
    decrease is ignored rather than invented as a re-unlock.
    """
    instances = []
    position = collections.Counter()
    for index in range(len(steps)):
        shops = shop_observed_shops(steps, index, seat)
        if shops is None:
            continue
        counts = collections.Counter(shops)
        for shop, count in counts.items():
            while position[shop] < count:
                instances.append({
                    "shop": shop,
                    "turn": index,
                    "day": _shop_day_of(_shop_observation(steps, index, seat), index),
                })
                position[shop] += 1
    return instances


def shop_days_active(timeline, last_day):
    """Map ``product -> set of days`` with at least one consuming shop unlocked.

    A shop unlocked on day D stays live for the rest of the match, so an instance
    contributes its products to every day from D through ``last_day``.
    """
    active = collections.defaultdict(set)
    for instance in timeline:
        products = SHOP_PRODUCTS.get(instance["shop"], ())
        for day in range(instance["day"], last_day + 1):
            for product in products:
                active[product].add(day)
    return dict(active)


# --------------------------------------------------------------------------
# Expected shop demand (mapping joined with town_shop_demand.csv)
# --------------------------------------------------------------------------

def shop_rule_instance_day_units(products, interval=SHOP_CONSUME_INTERVAL_TURNS):
    """Per-instance per-product units per day from the engine rule alone.

    One unit every ``interval`` turns, doubled when the shop has a single
    product: ``24 / 4 * (2 if single else 1)`` -> 12 (YARN_STORE, PET_CAFE) or 6.
    This is the mapping-only fallback when the mounted table is absent, and the
    cross-check for the table's own ``units_per_day_if_unlocked`` column.
    """
    interval = max(1, int(interval))
    multiplier = SHOP_SINGLE_PRODUCT_MULTIPLIER if len(products) == 1 else 1
    return multiplier * (SHOP_TURNS_PER_DAY // interval)


def shop_discover_demand_path(root="/kaggle/input"):
    """First mounted ``town_shop_demand.csv`` (searched recursively), else local.

    Kaggle mounts datasets flat and nested, so the search is recursive. The local
    engine-table cache is only a workstation fallback; ``None`` means neither was
    found and the caller must use the mapping-only fallback, flagged.
    """
    for path in sorted(glob.glob(os.path.join(root, "**", SHOP_DEMAND_FILENAME),
                                 recursive=True)):
        if os.path.isfile(path):
            return path
    for path in SHOP_LOCAL_DEMAND_PATHS:
        if os.path.isfile(path):
            return path
    return None


def _shop_as_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def shop_load_demand_table(path=None):
    """Load the mounted shop demand table, or fall back to the engine rule.

    The table is a join key, not a blind oracle: each ``source == "SHOP"`` row is
    matched to the engine's shop-product mapping and its
    ``units_per_day_if_unlocked`` is compared against :func:`shop_rule_instance_day_units`
    (which applies the single-product doubling). ``TOWN_CENTER`` rows are counted
    and deliberately EXCLUDED from shop demand. A missing table, a missing row or
    an unreadable file all degrade to the mapping-only rule and are flagged in the
    returned dict rather than fabricating a number.
    """
    table = {
        "present": False,
        "path": None,
        "rates": {},
        "shop_rows": 0,
        "town_rows": 0,
        "mismatches": [],
        "missing": [],
        "unknown": [],
    }
    if path is None:
        path = shop_discover_demand_path()
    if path and os.path.isfile(path):
        try:
            with open(path) as handle:
                reader = csv.DictReader(handle)
                for row in reader:
                    source = (row.get("source") or "").strip().upper()
                    if source == "TOWN_CENTER":
                        table["town_rows"] += 1
                        continue
                    if source != "SHOP":
                        continue
                    shop = (row.get("name") or "").strip()
                    product = (row.get("product") or "").strip()
                    if shop not in SHOP_PRODUCTS or product not in SHOP_PRODUCTS[shop]:
                        table["unknown"].append((shop, product))
                        continue
                    table["rates"][(shop, product)] = _shop_as_float(
                        row.get("units_per_day_if_unlocked"))
                    table["shop_rows"] += 1
            table["present"] = True
            table["path"] = path
        except OSError:
            table["path"] = path
    # Fill every mapping entry the table did not supply (or the whole table when
    # it is absent) from the engine rule, and record what was filled.
    for shop, products in SHOP_PRODUCTS.items():
        for product in products:
            if (shop, product) in table["rates"]:
                expected = float(shop_rule_instance_day_units(products))
                actual = table["rates"][(shop, product)]
                if abs(actual - expected) > 1e-9:
                    table["mismatches"].append(
                        (shop, product, actual, expected))
            else:
                table["rates"][(shop, product)] = float(shop_rule_instance_day_units(products))
                if table["present"]:
                    table["missing"].append((shop, product))
    return table


def shop_expected_demand_by_day(timeline, last_day, rates):
    """``day -> Counter(product -> expected shop units/day)``.

    Sums, per day, every live instance's per-product rate: ``rates`` from the
    mounted table (or the mapping-only rule). Only days where a product has at
    least one consuming instance appear, so a mean over these rows is the mean
    demand on ACTIVE days rather than a figure diluted by zero-shop days.
    """
    added = collections.defaultdict(collections.Counter)
    for instance in timeline:
        added[instance["day"]][instance["shop"]] += 1
    demand = {}
    running = collections.Counter()
    for day in range(last_day + 1):
        running.update(added.get(day) or {})
        counts = collections.defaultdict(float)
        for shop, instances in running.items():
            for product in SHOP_PRODUCTS.get(shop, ()):
                rate = rates.get((shop, product))
                if rate is None:
                    rate = float(shop_rule_instance_day_units(SHOP_PRODUCTS[shop]))
                counts[product] += instances * rate
        if counts:
            demand[day] = counts
    return demand


# --------------------------------------------------------------------------
# Executed sold units via holdings-drop accounting
# --------------------------------------------------------------------------

def shop_holdings(obs):
    """Units of every item held: ``private.shed`` + every farmer inventory.

    The sum is the right lens because internal shed<->inventory moves cancel
    inside it; a drop therefore means units LEFT the farm.
    """
    private = obs.get("private") if isinstance(obs, dict) else None
    held = collections.Counter()
    if not isinstance(private, dict):
        return held
    shed = private.get("shed")
    if isinstance(shed, dict):
        for item, units in shed.items():
            if isinstance(units, (int, float)) and not isinstance(units, bool):
                held[str(item)] += int(units)
    inventories = private.get("inventories")
    if isinstance(inventories, (list, tuple)):
        for inv in inventories:
            if isinstance(inv, dict):
                for item, units in inv.items():
                    if isinstance(units, (int, float)) and not isinstance(units, bool):
                        held[str(item)] += int(units)
    return held


def shop_requested_sell_products(action):
    """Products with a ``SELL`` order in one turn's action (the request only)."""
    products = set()
    if not isinstance(action, dict):
        return products
    for order in action.get("market") or []:
        if (isinstance(order, (list, tuple)) and len(order) >= 2
                and order[0] == "SELL"):
            products.add(str(order[1]))
    return products


def shop_executed_sales(steps, seat):
    """Executed sold units per product per day, from holdings drops.

    For each consecutive observable pair ``(i-1, i)`` the holdings of ``seat`` are
    differenced; a POSITIVE drop is executed market volume. The turn's action
    (which produced ``observation[i]``) must request ``SELL`` for that product,
    which is what separates a real fill from two other drops: the town-centre /
    feed consumption and the end-of-day shed-overflow discard. A requested
    ``SELL`` with no drop is a request the market did not fill and counts zero.

    Returns ``{"by_day_product", "by_product", "dropped_unsold"}``. The unsold
    drops are kept as a diagnostic, never as sales.
    """
    by_day = collections.defaultdict(collections.Counter)
    by_product = collections.Counter()
    dropped_unsold = collections.Counter()
    for index in range(1, len(steps)):
        obs = _shop_observation(steps, index, seat)
        prev = _shop_observation(steps, index - 1, seat)
        if obs is None or prev is None:
            continue
        held_now = shop_holdings(obs)
        held_prev = shop_holdings(prev)
        state = _shop_state(steps, index, seat) or {}
        sells = shop_requested_sell_products(state.get("action"))
        day = _shop_day_of(obs, index)
        for product in set(held_prev) | set(held_now):
            drop = held_prev.get(product, 0) - held_now.get(product, 0)
            if drop <= 0:
                continue
            if product in sells:
                by_day[day][product] += drop
                by_product[product] += drop
            else:
                dropped_unsold[product] += drop
    return {
        "by_day_product": {day: collections.Counter(counts)
                           for day, counts in by_day.items()},
        "by_product": by_product,
        "dropped_unsold": dropped_unsold,
    }


# --------------------------------------------------------------------------
# Metrics
# --------------------------------------------------------------------------

def shop_window_summary(executed, active_days):
    """Shop-window share of a seat's executed units.

    For each shop-consuming product: units sold on days with at least one
    unlocked shop consuming it, over total executed units of that product. The
    aggregate is the same ratio over the summed numerator and denominator of all
    shop-consuming products that actually sold. A product with zero executed
    units is excluded from the denominator, not counted as a zero share.
    """
    by_day = executed["by_day_product"]
    per_product = {}
    shop_units = 0
    in_window = 0
    for product in SHOP_CONSUMING_PRODUCTS:
        total = executed["by_product"].get(product, 0)
        if total <= 0:
            continue
        days = set(active_days.get(product, ()))
        window_units = sum(counts.get(product, 0)
                           for day, counts in by_day.items() if day in days)
        per_product[product] = {
            "sold": total,
            "in_window": window_units,
            "share": window_units / total,
        }
        shop_units += total
        in_window += window_units
    non_shop_units = sum(executed["by_product"].get(product, 0)
                         for product in SHOP_NON_PRODUCTS)
    return {
        "per_product": per_product,
        "aggregate": {
            "sold": shop_units,
            "in_window": in_window,
            "share": (in_window / shop_units) if shop_units else None,
        },
        "shop_units": shop_units,
        "non_shop_units": non_shop_units,
    }


def shop_yarn_split(executed, active_days):
    """The tape's YARN gate: wool units on YARN_STORE-active days vs other days."""
    yarn_days = set(active_days.get(SHOP_YARN_PRODUCT, ()))
    on_units = off_units = 0
    for day, counts in executed["by_day_product"].items():
        units = counts.get(SHOP_YARN_PRODUCT, 0)
        if units <= 0:
            continue
        if day in yarn_days:
            on_units += units
        else:
            off_units += units
    total = on_units + off_units
    return {
        "has_yarn": bool(yarn_days),
        "on": on_units,
        "off": off_units,
        "share_on": (on_units / total) if total else None,
    }


def shop_extract_match(steps, teams=None):
    """Per-match shop extraction, both seats of the SAME replay.

    Every match qualifies (unlike the economic study's primary cohort): the
    comparison is winner against loser of one replay. Raises :class:`ShopSkip`
    for a truncated replay it cannot read.
    """
    if not isinstance(steps, (list, tuple)) or len(steps) < 2:
        raise ShopSkip("truncated: fewer than two steps")
    seats = {}
    for seat in (0, 1):
        timeline = shop_timeline(steps, seat)
        executed = shop_executed_sales(steps, seat)
        last_day = shop_observed_last_day(steps, seat)
        if last_day is None:
            last_day = 0
        active = shop_days_active(timeline, last_day)
        seats[seat] = {
            "seat": seat,
            "timeline": timeline,
            "last_day": last_day,
            "executed": executed,
            "active_days": active,
            "window": shop_window_summary(executed, active),
            "yarn": shop_yarn_split(executed, active),
            "money": shop_final_money(steps, seat),
        }
    money0 = seats[0]["money"]
    money1 = seats[1]["money"]
    winner, loser = (0, 1) if money0 >= money1 else (1, 0)
    return {
        "teams": [str(team) for team in (teams or [])],
        "seats": seats,
        "winner": winner,
        "loser": loser,
        "money": [money0, money1],
        "tie": money0 == money1,
    }


# --------------------------------------------------------------------------
# Bounded aggregation
# --------------------------------------------------------------------------

def shop_new_summary(demand_table=None):
    """Accumulator for the shop analysis across one full-parse pass.

    Only scalars and short per-seat lists survive: no replay or timeline object
    is retained, so a 6,000-match pass stays a few MB.
    """
    return {
        "parsed": 0,
        "matches": 0,
        "seats": 0,
        "ties": 0,
        "skipped": collections.Counter(),
        "demand_table": demand_table if demand_table is not None
        else shop_load_demand_table(),
        "matched_teams": set(),
        "final_instances": [],
        "shop_instance_counts": collections.Counter(),
        "shop_seats": collections.Counter(),
        "shop_unlock_days": collections.defaultdict(list),
        "duplicate_seats": 0,
        "expected_product": collections.defaultdict(lambda: [0.0, 0]),
        "sold_product": collections.defaultdict(list),
        "window_product": collections.defaultdict(
            lambda: {"sold": [], "share": []}),
        "shop_units": [],
        "non_shop_units": [],
        "window": {
            "both_defined": 0, "one_defined": 0, "none_defined": 0,
            "winner_more": 0, "loser_more": 0, "tie": 0,
            "winner_share": [], "loser_share": [],
            "winner_sold": [], "loser_sold": [],
        },
        "split": {
            "winner_shop": [], "loser_shop": [],
            "winner_non": [], "loser_non": [],
            "winner_more_shop": 0, "loser_more_shop": 0, "tie_shop": 0,
            "winner_more_non": 0, "loser_more_non": 0, "tie_non": 0,
        },
        "yarn": {
            "seats_with_yarn": 0, "seats_without": 0,
            "on_units": 0, "off_units": 0, "on_vals": [], "off_vals": [],
        },
    }


def _shop_fold_seat(summary, profile):
    summary["seats"] += 1
    timeline = profile["timeline"]
    summary["final_instances"].append(len(timeline))
    per_shop = collections.Counter(instance["shop"] for instance in timeline)
    summary["shop_instance_counts"].update(per_shop)
    if any(count >= 2 for count in per_shop.values()):
        summary["duplicate_seats"] += 1
    for shop in per_shop:
        summary["shop_seats"][shop] += 1
    for instance in timeline:
        summary["shop_unlock_days"][instance["shop"]].append(instance["day"])

    rates = summary["demand_table"]["rates"]
    for counts in shop_expected_demand_by_day(timeline, profile["last_day"], rates).values():
        for product, units in counts.items():
            cell = summary["expected_product"][product]
            cell[0] += units
            cell[1] += 1
    for product, sold in profile["executed"]["by_product"].items():
        summary["sold_product"][product].append(sold)

    window = profile["window"]
    summary["shop_units"].append(window["shop_units"])
    summary["non_shop_units"].append(window["non_shop_units"])
    for product, info in window["per_product"].items():
        summary["window_product"][product]["sold"].append(info["sold"])
        summary["window_product"][product]["share"].append(info["share"])

    yarn = profile["yarn"]
    summary["yarn"]["seats_with_yarn" if yarn["has_yarn"] else "seats_without"] += 1
    summary["yarn"]["on_units"] += yarn["on"]
    summary["yarn"]["off_units"] += yarn["off"]
    if yarn["on"] + yarn["off"] > 0:
        summary["yarn"]["on_vals"].append(yarn["on"])
        summary["yarn"]["off_vals"].append(yarn["off"])


def shop_fold_match(summary, match):
    """Fold one :func:`shop_extract_match` result into ``summary``.

    Both seats are counted in the per-seat tables. The winner/loser rows are
    WITHIN one replay and are skipped on a tied final bank, because a tie has no
    direction to report.
    """
    summary["parsed"] += 1
    seats = match["seats"]
    for team in match.get("teams") or ():
        summary["matched_teams"].add(team)
    for seat in (0, 1):
        profile = seats.get(seat)
        if profile is not None:
            _shop_fold_seat(summary, profile)
    winner = seats.get(match["winner"])
    loser = seats.get(match["loser"])
    if winner is None or loser is None:
        return
    if match.get("tie"):
        summary["ties"] += 1
        return
    summary["matches"] += 1

    window = summary["window"]
    winner_share = winner["window"]["aggregate"]["share"]
    loser_share = loser["window"]["aggregate"]["share"]
    if winner_share is not None and loser_share is not None:
        window["both_defined"] += 1
        window["winner_share"].append(winner_share)
        window["loser_share"].append(loser_share)
        window["winner_sold"].append(winner["window"]["aggregate"]["sold"])
        window["loser_sold"].append(loser["window"]["aggregate"]["sold"])
        if winner_share > loser_share:
            window["winner_more"] += 1
        elif loser_share > winner_share:
            window["loser_more"] += 1
        else:
            window["tie"] += 1
    elif winner_share is None and loser_share is None:
        window["none_defined"] += 1
    else:
        window["one_defined"] += 1

    split = summary["split"]
    winner_shop = winner["window"]["shop_units"]
    loser_shop = loser["window"]["shop_units"]
    winner_non = winner["window"]["non_shop_units"]
    loser_non = loser["window"]["non_shop_units"]
    split["winner_shop"].append(winner_shop)
    split["loser_shop"].append(loser_shop)
    split["winner_non"].append(winner_non)
    split["loser_non"].append(loser_non)
    if winner_shop > loser_shop:
        split["winner_more_shop"] += 1
    elif loser_shop > winner_shop:
        split["loser_more_shop"] += 1
    else:
        split["tie_shop"] += 1
    if winner_non > loser_non:
        split["winner_more_non"] += 1
    elif loser_non > winner_non:
        split["loser_more_non"] += 1
    else:
        split["tie_non"] += 1


# --------------------------------------------------------------------------
# Bounded render
# --------------------------------------------------------------------------

def _shop_median(values):
    return statistics.median(values) if values else None


def _shop_count(value):
    return "-" if value is None else f"{value:,.0f}"


def _shop_share(value):
    return "-" if value is None else f"{value:.1%}"


def _shop_day(value):
    return "-" if value is None else f"{value:.0f}"


def shop_render_report(summary, date):
    """Return the bounded (< 20 kB) text report for one dataset date.

    Every row prints the n behind it, every skip reason is listed, and the
    closing note states the within-game caveat plus the single-dataset-date
    boundary. Descriptive only: no causal claim.
    """
    lines = []
    add = lines.append
    table = summary["demand_table"]
    add("=" * 96)
    add(f"SHOP-WINDOW DEMAND ANALYSIS - dataset date {date} (one day per run)")
    add("=" * 96)
    add(f"replays parsed        : {summary['parsed']:,}")
    add(f"matches (both seats)  : {summary['matches']:,} "
        "(winner = higher final farm.money, same replay)")
    add(f"seats profiled        : {summary['seats']:,}")
    add(f"distinct teams        : {len(summary['matched_teams']):,} "
        "(shared across replays; rows are not independent people)")
    add(f"tied final bank       : {summary['ties']:,} "
        "(no direction; excluded from winner/loser rows)")
    if table.get("present"):
        add(f"shop demand table     : {table['path']} "
            f"({table['shop_rows']} SHOP rows joined)")
    else:
        add("shop demand table     : MISSING - mapping-only fallback (FLAGGED)")
    add(f"town_center rows      : {table['town_rows']:,} seen, "
        "EXCLUDED from shop demand on purpose (1 unit/product/24 turns)")
    if table["mismatches"]:
        add(f"table vs engine rule  : {len(table['mismatches'])} rate mismatch(es) "
            "FLAGGED; the table value is used")
    else:
        add("table vs engine rule  : every joined rate matches the "
            "single-product doubling rule (24/4 * 2 if single -> 12 else 6)")
    if table["missing"]:
        add(f"mapping rows the table omitted: {len(table['missing'])} "
            "filled from the engine rule (flagged)")
    if table["unknown"]:
        add(f"table rows outside the engine mapping: {len(table['unknown'])} ignored")
    if summary["skipped"]:
        add(f"skipped replays       : {sum(summary['skipped'].values()):,}")
        for reason, count in summary["skipped"].most_common():
            add(f"  {count:>7,}  {reason}")
    else:
        add("skipped replays       : 0")

    add("")
    add("--- OBSERVED SHOP TIMELINE (town.unlocked_shops each turn; duplicates kept) ---")
    add(f"{'shop':<18}{'instances':>10}{'seats':>7}{'first day med':>15}{'n':>7}")
    for shop in sorted(SHOP_PRODUCTS):
        days = summary["shop_unlock_days"].get(shop) or []
        add(f"{shop:<18}{summary['shop_instance_counts'].get(shop, 0):>10,}"
            f"{summary['shop_seats'].get(shop, 0):>7,}"
            f"{_shop_day(_shop_median(days)):>15}{len(days):>7,}")
    counts = summary["final_instances"]
    add(f"final live instances per seat: median {_shop_count(_shop_median(counts))}"
        f" (min {_shop_count(min(counts) if counts else None)},"
        f" max {_shop_count(max(counts) if counts else None)},"
        f" cap {SHOP_MAX_INSTANCES}, n={len(counts):,} seats)")
    add(f"seats with a duplicated instance: {summary['duplicate_seats']:,}/"
        f"{summary['seats']:,} (draw is with replacement; copies consume independently)")
    if not counts:
        add("No seat exposed a readable town.unlocked_shops list; nothing to table.")

    add("")
    add("--- EXPECTED SHOP DEMAND PER PRODUCT (mapping x town_shop_demand.csv) ---")
    add(f"{'product':<14}{'active seat-days':>18}{'mean units/day':>16}"
        f"{'instances rx':>14}")
    any_expected = False
    for product in SHOP_CONSUMING_PRODUCTS:
        cell = summary["expected_product"].get(product)
        if not cell or not cell[1]:
            continue
        any_expected = True
        add(f"{product:<14}{cell[1]:>18,}{cell[0] / cell[1]:>16.2f}"
            f"{'':>14}")
    if not any_expected:
        add("no shop-consuming product had an unlocked consumer; row omitted")
    add("'mean units/day' is over seat-days with >=1 consuming shop unlocked; the")
    add("town-centre baseline is excluded, so these are shop-only rates.")

    add("")
    add("--- EXECUTED UNITS AND SHOP-WINDOW SHARE (per seat, within-game) ---")
    add(f"{'product':<14}{'seats sold>0':>13}{'sold med':>10}{'share med':>11}"
        f"{'n share':>9}")
    for product in SHOP_CONSUMING_PRODUCTS + SHOP_NON_PRODUCTS:
        sold = summary["sold_product"].get(product) or []
        if not sold:
            continue
        info = summary["window_product"].get(product) or {"sold": [], "share": []}
        add(f"{product:<14}{len(sold):>13,}{_shop_count(_shop_median(sold)):>10}"
            f"{_shop_share(_shop_median(info['share'])):>11}{len(info['share']):>9,}")
    aggregate_shares = []
    for product in SHOP_CONSUMING_PRODUCTS:
        info = summary["window_product"].get(product)
        if info and info["sold"]:
            aggregate_shares.extend(info["share"])
    window = summary["window"]
    add(f"aggregate shop-consuming share per seat: median "
        f"{_shop_share(_shop_median(aggregate_shares))} (n={len(aggregate_shares):,} seats "
        "with >=1 unit sold of a shop-consuming product)")
    add(f"executed units per seat: shop-consuming products median "
        f"{_shop_count(_shop_median(summary['shop_units']))}, non-shop products median "
        f"{_shop_count(_shop_median(summary['non_shop_units']))} "
        f"(n={len(summary['shop_units']):,} seats)")

    add("")
    add("--- WITHIN-GAME: WINNER vs LOSER OF THE SAME REPLAY ---")
    add(f"matches with both window shares defined: {window['both_defined']:,}"
        f" (one defined {window['one_defined']:,}, none {window['none_defined']:,})")
    if window["both_defined"]:
        add(f"winner has the HIGHER shop-window share: {window['winner_more']:,}/"
            f"{window['both_defined']:,} = "
            f"{window['winner_more'] / window['both_defined']:.1%} "
            f"(loser higher {window['loser_more']:,}, ties {window['tie']:,})")
        add(f"median winner share {_shop_share(_shop_median(window['winner_share']))} vs loser "
            f"{_shop_share(_shop_median(window['loser_share']))}")
    else:
        add("no match has both seats' shop-window share defined; row omitted")
    split = summary["split"]
    if split["winner_shop"]:
        add(f"shop-product units: winner median {_shop_count(_shop_median(split['winner_shop']))}"
            f" vs loser {_shop_count(_shop_median(split['loser_shop']))}; winner has more "
            f"{split['winner_more_shop']:,}/{summary['matches']:,} = "
            f"{split['winner_more_shop'] / max(1, summary['matches']):.1%} "
            f"(n={len(split['winner_shop']):,} matches)")
        add(f"non-shop units   : winner median {_shop_count(_shop_median(split['winner_non']))}"
            f" vs loser {_shop_count(_shop_median(split['loser_non']))}; winner has more "
            f"{split['winner_more_non']:,}/{summary['matches']:,} = "
            f"{split['winner_more_non'] / max(1, summary['matches']):.1%}")
    else:
        add("no non-tied match to compare; row omitted")

    add("")
    add("--- YARN GATE (the tape's one observed shop route switch) ---")
    yarn = summary["yarn"]
    add(f"seats with >=1 YARN_STORE instance: {yarn['seats_with_yarn']:,}; "
        f"without: {yarn['seats_without']:,}")
    total_wool = yarn["on_units"] + yarn["off_units"]
    add(f"wool executed units: on YARN-active days {yarn['on_units']:,}, "
        f"other days {yarn['off_units']:,}, total {total_wool:,}")
    if total_wool:
        add(f"share of wool sold on YARN-active days: "
            f"{yarn['on_units'] / total_wool:.1%} "
            f"(denominator {total_wool:,} units; n="
            f"{len(yarn['on_vals']):,} seats with >0 wool sold)")
        add(f"per-seat median wool units: on-days "
            f"{_shop_count(_shop_median(yarn['on_vals']))}, off-days "
            f"{_shop_count(_shop_median(yarn['off_vals']))}")
    else:
        add("no executed wool sales observed; YARN split not computable")

    add("")
    add("Reading rules: every row prints its own n. Executed units are HOLDINGS DROPS")
    add("on a turn that requested SELL, never the requested quantity (SELL 10000 is a")
    add("request the market may not fill). Expected shop demand comes from the mounted")
    add("engine table joined to the shop-product mapping, with the single-product")
    add("doubling applied; the town-centre baseline is excluded. The within-game")
    add("control removes shared shops and market, not policy, skill or selection, so")
    add("this is DESCRIPTIVE: repeated teams make rows non-independent and one dataset")
    add(f"date ({date}) supports no cross-day claim.")
    return "\n".join(lines)


## One-day guard - fail closed before any replay is parsed

The economic study runs on exactly one daily episode dataset. The metadata
attaches one, but this cell re-checks the mounted tree, because a dataset list is
easy to edit and a pooled run is not an event study. Zero or more than one daily
source stops the notebook with `SystemExit`; index, engine and reference-agent
datasets are not daily sources and are ignored.

In [ ]:
# --- ONE-DAY GUARD: fail closed BEFORE any replay is parsed ---
# The economic study analyses exactly one daily episode dataset per run. The
# metadata attaches one, but a stale or mis-edited dataset_sources list could
# mount several days, so the guard re-checks the mounted tree itself. The index
# dataset, engine tables and reference agents are not daily sources and are
# ignored on purpose.
candidate_dirs = set()
for _manifest in manifests:
    candidate_dirs.add(os.path.dirname(_manifest))
for _path in glob.glob("/kaggle/input/**/kaggriculture-episodes-*", recursive=True):
    candidate_dirs.add(_path)
try:
    DATA_DATE, DATA_DIR = require_exactly_one_day(sorted(candidate_dirs))
except OneDayGuardError as exc:
    raise SystemExit("FATAL one-day guard: " + str(exc))
print(f"one-day guard OK: dataset_date={DATA_DATE}")
print(f"  daily source : {DATA_DIR}")
print(f"  dirs scanned : {len(candidate_dirs)} (non-daily sources ignored)")


## Tier 1 - a prefix scan of every replay (~300 MB, not 1.19 TB)

`info.TeamNames` and `seed` sit at one end or the other of the JSON, so reading
~12 kB per file yields a league-wide team index. This is the part that scales to
many days: cost grows with file count, not with 35 MB per episode.

In [ ]:
import multiprocessing as mp

MAX_EPISODES = 6000   # keep a session comfortable; raise it once the cost is known

def scan_one(path):
    # Tier 1: read only the head and tail of a replay. info.TeamNames and seed sit
    # at one end or the other, so ~12 kB per file answers team questions for the
    # whole league instead of parsing 35 MB.
    try:
        with open(path, "rb") as handle:
            head = handle.read(4096)
            handle.seek(max(0, handle.seek(0, 2) - 8192))
            tail = handle.read(8192)
        blob = (head + b" " + tail).decode("utf-8", "replace")
        match = re.search(r'"TeamNames"\s*:\s*\[(.*?)\]', blob, re.S)
        if not match:
            return None
        names = json.loads("[" + match.group(1) + "]")   # decodes \uXXXX escapes
        seed = re.search(r'"seed"\s*:\s*(-?\d+)', blob)
        return (os.path.basename(path), names, int(seed.group(1)) if seed else None)
    except (OSError, ValueError):
        return None

sample = episode_files[:MAX_EPISODES]
started = time.time()
scanned = []
if sample:
    with mp.Pool(min(8, os.cpu_count() or 1)) as pool:
        for result in pool.imap_unordered(scan_one, sample, chunksize=32):
            if result:
                scanned.append(result)
    elapsed = time.time() - started
    rate = len(sample) / max(elapsed, 1e-9)
    print(f"scanned {len(sample):,} replays in {elapsed:.1f}s ({rate:,.0f}/s), "
          f"{len(scanned):,} parsed")
    print(f"projected for the whole league ({38454:,}): "
          f"{38454 / max(rate, 1e-9) / 60:.1f} minutes")
    print(f"bytes read: ~{len(sample) * 12 / 1e6:.0f} MB "
          f"(versus ~{len(sample) * 35 / 1e6:.0f} MB if fully parsed)")

In [ ]:
bands = {}
episode_day = {}
for row in episodes:
    episode_day[row["episode_id"]] = row["_source"]
    if row.get("avg_score"):
        bands[row["episode_id"]] = float(row["avg_score"])

team_days = collections.Counter()
team_games = collections.Counter()
team_scores = collections.defaultdict(list)
for name, names, seed in scanned:
    episode_id = name.split(".")[0]
    score = bands.get(episode_id)
    day = episode_day.get(episode_id, "?")   # dict lookup, not a linear scan per file
    for team in names:
        team_games[team] += 1
        team_days[(team, day)] += 1
        if score:
            team_scores[team].append(score)

team_day_count = collections.Counter(team for team, _ in team_days)

print(f"distinct teams seen: {len(team_games):,}")
print(f"\n{'team':<30}{'games':>7}{'days':>6}{'mean band':>12}")
for team, count in team_games.most_common(15):
    scores = team_scores.get(team, [])
    print(f"{team[:30]:<30}{count:>7}{team_day_count[team]:>6}"
          f"{(sum(scores) / len(scores) if scores else 0):>12.0f}")

print("\nThe corpus is published as daily TOP episodes, a far higher rating")
print("band than our current submissions whatever their scores. Comparing our games")
print("against that band compares populations, not strategies; only within-game")
print("comparisons of the two seats of one match are safe.")

## Tier 2 - full-replay BEHAVIOUR, always with the within-game control

The tier the notebook's own next-steps list was missing: `income_breakdown.py` /
`sale_context.py` / `sell_timing.py` ported so behaviour analyses run over a large
sample, always with both seats of the same match beside every cross-team number.

**Why the within-game control is not a stylistic preference.** The corpus is
published as the day's TOP episodes, so it is selection-biased on the OUTCOME: no
episode here sits in our rating band. Comparing a top game with a random one is
confounded, and this project was already burned by exactly that (a +$25k
"opportunity" that turned out to be an artefact). Inside one match both seats face
the same shops, the same town drain and the same market, so winner-against-loser
within a match is the only safe axis. The analysis profiles BOTH seats of every
replay and compares the winner with the loser of that same game - never one corpus
band against another.

**Cost, and how it stays inside memory.** One replay is ~34 MB, a day is ~21 GB and
the mounted corpus is ~190 GB. Each replay is opened, profiled and released before
the next one is read, and only scalars survive an iteration. The cell prints the
measured rate, the number processed, the number skipped **with the reason**, and a
projection for the whole corpus, so a truncated sample can never pass for a clean
result. `MAX_REPLAYS_T2` bounds the work, so a slow run still prints a report
instead of timing out.

**The report leads with the NEWEST mounted day, alone.** The local 608-episode
baseline came from ONE published day, recorded in the report's own constant. If the
mounted day is a different date, the report treats it as an out-of-sample
replication and prints a CONFIRMS / SHRINKS / CONTRADICTS verdict. If the mounted day
IS that baseline's date, it says so explicitly, labels the comparison same-sample
calibration and prints no verdict, because a day measured against itself is not
independent evidence and implies no cross-day robustness. A single mounted day skips
the pool entirely instead of double-counting that one day, and a per-day table
checks stability across days only when more than one is attached.

**One parse feeds two reports.** The same loop that already holds each parsed
replay also feeds the daily/event-time extractor, so the economic study does not
pay a second ~590-replay read.

In [ ]:
import resource
import statistics

# Tier 2 - full-replay BEHAVIOUR analysis, always with the within-game control.
#
# This is the tier the notebook's own next-steps list was missing: income_breakdown
# / sale_context / sell_timing ported so behaviour analyses run over a large sample,
# always with both seats of the same match beside every cross-team number.
#
# Why the within-game control is NOT a stylistic preference: the corpus is published
# as the day's TOP episodes, so it is selection-biased on the OUTCOME (no episode
# here sits in our rating band). Comparing a top game with a random one is
# confounded, and this project was already burned by exactly that (a +$25k
# "opportunity" that turned out to be an artefact). Inside one match both seats face
# the same shops, the same town drain and the same market, so winner-against-loser
# WITHIN a match is the only safe axis. Every number below is computed that way.
#
# Memory and IO are the whole problem: one replay is ~34 MB, a day is ~21 GB and the
# mounted corpus is ~190 GB. So ONE replay is opened, profiled and released before
# the next is read, and only scalars survive the iteration - nothing here holds a
# day, or a thousand parsed replays, in memory. It stays single process on purpose:
# a 4-way pool would hold four replays and four parsed object graphs at once, and
# the mounted read is the bottleneck anyway.

TURNS_PER_DAY = 24

# Taken verbatim from scripts/corpus_within_game.py, so the numbers stay comparable
# with the local 608-episode result.
T2_METRICS = (
    "hires", "land", "seed_units", "animals_bought", "plants", "harvests",
    "waters", "feeds", "fertilizes", "collects", "builds",
    "sells", "sell_units", "sell_lot_median", "distinct_sell_days",
    "peak_crop_tiles", "peak_animals", "quadrants",
)

MAX_REPLAYS_T2 = 6000   # a bound, so a run that is too slow still prints a report


def t2_day(path):
    match = re.search(r"kaggriculture-episodes-(\d{4}-\d{2}-\d{2})", path)
    if match:
        return match.group(1)
    # Never assume the layout: a dataset mounted under another name still has a
    # parent directory, and a file whose day cannot be recovered is reported under
    # that rather than silently dropped.
    return os.path.basename(os.path.dirname(path)) or "?"


def t2_verbs(action):
    # Farm verbs this agent asks for in one turn. Copied from the local script.
    if not isinstance(action, dict):
        return []
    units = [action.get("farmer"), *(action.get("hands") or [])]
    return [unit[0] for unit in units if isinstance(unit, (list, tuple)) and unit]


def t2_sum_orders(action, verb):
    # Units this action asks for under verb, summed over its market orders.
    total = 0
    if isinstance(action, dict):
        for order in action.get("market") or []:
            if (isinstance(order, (list, tuple)) and len(order) >= 3
                    and order[0] == verb):
                try:
                    total += int(order[2])
                except (TypeError, ValueError):
                    continue
    return total


def t2_profile_seat(steps, seat):
    # Ported from scripts/corpus_within_game.py:profile_seat. Behaviour is identical,
    # including the median sell lot over that seat's SELL orders and the peaks over
    # the whole match; changing it would break comparability with the local result.
    m = collections.Counter()
    lots = []
    sell_days = set()
    peak_crops = peak_animals = peak_hands = 0
    quadrants = 1
    bank = 0.0

    for index in range(1, len(steps)):
        entry = steps[index][seat] if seat < len(steps[index]) else {}
        obs = entry.get("observation") or {}
        action = entry.get("action")
        day = obs.get("day", index // TURNS_PER_DAY)

        for verb in t2_verbs(action):
            if verb == "PLANT":
                m["plants"] += 1
            elif verb == "HARVEST":
                m["harvests"] += 1
            elif verb == "WATER":
                m["waters"] += 1
            elif verb == "FEED":
                m["feeds"] += 1
            elif verb == "FERTILIZE":
                m["fertilizes"] += 1
            elif verb == "COLLECT_FERTILIZER":
                m["collects"] += 1
            elif verb in ("BUILD_COOP", "BUILD_PASTURE"):
                m["builds"] += 1

        if isinstance(action, dict):
            for order in action.get("market") or []:
                if not isinstance(order, (list, tuple)) or not order:
                    continue
                if order[0] == "HIRE":
                    m["hires"] += 1
                elif order[0] == "BUY_LAND":
                    m["land"] += 1
                elif order[0] == "BUY_ANIMAL":
                    m["animals_bought"] += 1
                elif order[0] == "SELL" and len(order) >= 3:
                    m["sells"] += 1
                    sell_days.add(day)
                    try:
                        quantity = int(order[2])
                    except (TypeError, ValueError):
                        continue
                    lots.append(quantity)
                    m["sell_units"] += quantity
            m["seed_units"] += t2_sum_orders(action, "BUY_SEED")

        crops = animals = 0
        farm = (obs.get("farms") or [{}])[obs.get("player", seat)] or {}
        for row in farm.get("tiles") or []:
            for cell in row:
                if not isinstance(cell, dict):
                    continue
                if cell.get("animal"):
                    animals += 1
                elif cell.get("kind") == "PLANT":
                    crops += 1
        peak_crops = max(peak_crops, crops)
        peak_animals = max(peak_animals, animals)
        peak_hands = max(peak_hands, len(farm.get("hands") or []))
        quadrants = len(farm.get("unlocked_quadrants") or [])
        bank = farm.get("money", bank)

    return {
        **m,
        "sell_lot_median": statistics.median(lots) if lots else 0,
        "distinct_sell_days": len(sell_days),
        "peak_crop_tiles": peak_crops,
        "peak_animals": peak_animals,
        "peak_hands": peak_hands,
        "quadrants": quadrants,
        "bank": bank,
    }


def t2_unlock_days(steps, seat):
    # Ported from scripts/corpus_quadrant_timing.py:unlock_days. farm[
    # unlocked_quadrants] starts as ["NW"], so length 2 is the first paid quadrant
    # (NE), 3 is the SW and 4 is the SE. Returns the day each unlocked and the bank
    # gap between the buyer and the OPPONENT at the moment the 4th was bought.
    days = {}
    gap_at_four = None
    bank = 0.0
    for index in range(1, len(steps)):
        entry = steps[index][seat] if seat < len(steps[index]) else {}
        obs = entry.get("observation") or {}
        farm = (obs.get("farms") or [{}])[obs.get("player", seat)] or {}
        count = len(farm.get("unlocked_quadrants") or [])
        day = obs.get("day", index // TURNS_PER_DAY)
        bank = farm.get("money", bank)
        if count >= 2 and 2 not in days:
            days[2] = day
        if count >= 3 and 3 not in days:
            days[3] = day
        if count >= 4 and 4 not in days:
            days[4] = day
            other = steps[index][1 - seat] if 1 - seat < len(steps[index]) else {}
            oobs = other.get("observation") or {}
            ofarm = (oobs.get("farms") or [{}])[oobs.get("player", 1 - seat)] or {}
            gap_at_four = farm.get("money", 0) - ofarm.get("money", 0)
    return days, gap_at_four, bank


def t2_new_bucket():
    # Only scalars and short lists; a bucket for 6,000 matches is a few MB at most.
    return {
        "games": 0,
        "seat_quad": collections.defaultdict(lambda: [0, 0]),   # quadrants -> [wins, played]
        "diff_n": 0, "diff_more": 0,
        "unlock": collections.defaultdict(list),
        "four_gap": [],
        "winner_vals": collections.defaultdict(list),
        "loser_vals": collections.defaultdict(list),
        "winner_more": collections.Counter(),
        "winner_less": collections.Counter(),
        "margins": [], "relative": [],
    }


def t2_fold(bucket, a, b):
    # One fold per MATCH: both seats are profiled, then winner against loser of that
    # same game. No cross-game comparison enters this function.
    bucket["games"] += 1
    if a["bank"] >= b["bank"]:
        winner, loser = a, b
    else:
        winner, loser = b, a

    for prof, other in ((a, b), (b, a)):
        bucket["seat_quad"][prof["quadrants"]][1] += 1
        if prof["bank"] > other["bank"]:
            bucket["seat_quad"][prof["quadrants"]][0] += 1
        for quad, day in prof["days"].items():
            bucket["unlock"][quad].append(day)
        if 4 in prof["days"]:
            bucket["four_gap"].append(prof["gap_at_four"])

    if a["quadrants"] != b["quadrants"]:
        bucket["diff_n"] += 1
        more = a if a["quadrants"] > b["quadrants"] else b
        other = b if more is a else a
        if more["bank"] > other["bank"]:
            bucket["diff_more"] += 1

    for metric in T2_METRICS:
        wv = winner.get(metric, 0) or 0
        lv = loser.get(metric, 0) or 0
        bucket["winner_vals"][metric].append(wv)
        bucket["loser_vals"][metric].append(lv)
        if wv > lv:
            bucket["winner_more"][metric] += 1
        elif wv < lv:
            bucket["winner_less"][metric] += 1

    gap = abs(a["bank"] - b["bank"])
    bucket["margins"].append(gap)
    bucket["relative"].append(gap / max(1.0, max(a["bank"], b["bank"])))


by_day = collections.defaultdict(list)
for path in episode_files:
    if t2_day(path) != "?":
        by_day[t2_day(path)].append(path)
t2_days = sorted(by_day)
# The full-parse pass must see exactly the day the runtime guard resolved; a
# mismatch means the sample is not the one the report names, so it stops.
try:
    require_expected_day(t2_days, DATA_DATE)
except OneDayGuardError as exc:
    raise SystemExit("FATAL one-day guard: " + str(exc))

# Round-robin across days, so a MAX_REPLAYS_T2 cut still covers every attached day
# instead of exhausting one of them and calling the result representative.
plan = []
cursors = {day: 0 for day in t2_days}
while True:
    progressed = False
    for day in t2_days:
        if cursors[day] < len(by_day[day]):
            plan.append((day, by_day[day][cursors[day]]))
            cursors[day] += 1
            progressed = True
    if not progressed:
        break

# One day mounted means every replay in it is the sample: silently slicing at the
# cap would shrink the day while the report still named it.
try:
    require_within_cap(len(plan), MAX_REPLAYS_T2, DATA_DATE)
except OneDayGuardError as exc:
    raise SystemExit("FATAL one-day guard: " + str(exc))

sample = plan[:MAX_REPLAYS_T2]
t2_newest = t2_days[-1] if t2_days else "?"
league_total = (sum(int(row["episode_count"]) for row in index_rows)
                if index_rows else 38454)
print(f"tier-2 sample : {len(sample):,} replays "
      f"(mounted {len(plan):,} across {len(t2_days)} days, cap {MAX_REPLAYS_T2:,})")
print(f"tier-2 days   : {', '.join(t2_days)}")
print(f"newest day    : {t2_newest}")


# One bucket per day as well as the pool: the per-day table is what shows whether the
# effect is stable or only tracks the field's strength on one day, and it costs a few
# scalars per match.
t2_pool = t2_new_bucket()
t2_day_buckets = {day: t2_new_bucket() for day in t2_days}
t2_skipped = collections.Counter()
t2_processed = collections.Counter()
t2_done = 0
# Economic accumulator filled in the SAME pass: the replay is already parsed and
# in memory, so the daily/event-time extraction costs no extra read.
econ_summary = new_summary()
# Shop-window accumulator in the SAME pass, with the mounted engine table loaded
# once. A missing table degrades to the shop-product mapping, FLAGGED in the
# report - never to a fabricated demand rate.
shop_demand_path = (find_all("town_shop_demand.csv") or [None])[0]
shop_table = shop_load_demand_table(shop_demand_path)
shop_summary = shop_new_summary(shop_table)
print(f"shop demand table : "
      + (shop_demand_path or "MISSING - mapping-only fallback (FLAGGED)"))
started = time.time()

for _cut, (_day, _path) in enumerate(sample, 1):
    try:
        with open(_path) as handle:
            data = json.load(handle)
    except (OSError, ValueError) as exc:
        t2_skipped["unreadable / not JSON (" + type(exc).__name__ + ")"] += 1
        continue
    steps = data.get("steps") if isinstance(data, dict) else data
    if not steps or len(steps) < 100 or any(len(s) < 2 for s in steps[:2]):
        t2_skipped["truncated: too few steps or seats"] += 1
        del data, steps
        continue
    statuses = (data.get("statuses") or []) if isinstance(data, dict) else []
    if any(str(s).upper().find("ERROR") >= 0 for s in statuses):
        t2_skipped["engine reported an ERROR status"] += 1
        del data, steps
        continue
    _teams = None
    _econ = None
    _shop = None
    try:
        a = t2_profile_seat(steps, 0)
        b = t2_profile_seat(steps, 1)
        for prof, seat in ((a, 0), (b, 1)):
            prof["days"], prof["gap_at_four"], _ = t2_unlock_days(steps, seat)
        _teams = ((data.get("info") or {}).get("TeamNames")
                  if isinstance(data, dict) else None)
        _econ = extract_match(steps, teams=_teams)
    except MatchSkip as exc:
        t2_skipped["econ: " + str(exc)] += 1
    except (AttributeError, IndexError, KeyError, TypeError) as exc:
        t2_skipped["malformed replay (" + type(exc).__name__ + ")"] += 1
        del data, steps
        continue
    # Shop-window extraction is independent of the econ cohort: EVERY match feeds
    # it, because the comparison is winner against loser of the same replay.
    try:
        _shop = shop_extract_match(steps, teams=_teams)
    except ShopSkip as exc:
        t2_skipped["shop: " + str(exc)] += 1
    except (AttributeError, IndexError, KeyError, TypeError) as exc:
        t2_skipped["shop malformed replay (" + type(exc).__name__ + ")"] += 1
    del data, steps   # release the ~34 MB replay before opening the next one

    t2_fold(t2_pool, a, b)
    t2_fold(t2_day_buckets[_day], a, b)
    if _econ is not None:
        fold_match(econ_summary, _econ)
    if _shop is not None:
        shop_fold_match(shop_summary, _shop)
    t2_processed[_day] += 1
    t2_done += 1

    if t2_done % 500 == 0:
        _elapsed = time.time() - started
        _rate = t2_done / max(_elapsed, 1e-9)
        print(f"  ... {t2_done:,} replays in {_elapsed:.0f}s ({_rate:.1f}/s), "
              f"{sum(t2_skipped.values()):,} skipped | projection for {len(plan):,}"
              f": {len(plan) / _rate / 60:.0f} min", flush=True)

elapsed = time.time() - started
rate = t2_done / max(elapsed, 1e-9)
peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print()
print(f"TIER 2 done: {t2_done:,} replays in {elapsed:.0f}s ({rate:.2f}/s), "
      f"peak RSS {peak_mb:.0f} MB")
print(f"  skipped: {sum(t2_skipped.values()):,}")
for reason, count in t2_skipped.most_common():
    print(f"    {count:>6,}  {reason}")
print(f"  processed per day: {dict(sorted(t2_processed.items()))}")
print(f"  projection: mounted corpus {len(plan):,} -> {len(plan) / rate / 60:.0f} min"
      f" | whole league {league_total:,} -> {league_total / rate / 3600:.1f} h"
      f" at this rate")
print(f"  bound MAX_REPLAYS_T2={MAX_REPLAYS_T2:,}: "
      + ("all mounted replays taken" if len(sample) == len(plan)
         else "sample CUT by the bound"))


### The report

Newest day first - its quadrant dose-response against the local measurement, either
as a same-sample calibration with no verdict (when the mounted day IS the local
baseline's own date) or as an out-of-sample replication with an explicit CONFIRMS /
SHRINKS / CONTRADICTS verdict (when it is a different day). Then the pool of all
attached days, skipped when only one day is mounted, then a per-day table that
states when cross-day stability cannot be inferred. Within each block: the
dose-response (wins and win rate
for seats ending with 2, 3 and 4 quadrants, and the share when the two seats
differ), the day each quadrant unlocked and the bank gap at the moment the 4th was
bought (that timing is what separates a cause from a symptom), the behaviour
separators with the winner-has-more share, and the margin distribution.

In [ ]:
def t2_report(bucket, title, full=True):
    print("=" * 84)
    print(title)
    print("=" * 84)
    games = bucket["games"]
    if not games:
        print("no usable matches in this bucket")
        print()
        return
    print(f"matches (both seats of the same game): {games:,}   seats: {2 * games:,}")
    print()

    print("--- QUADRANT DOSE-RESPONSE (the headline) ---")
    print(f"{'quadrants':>10}{'played':>10}{'wins':>8}{'win rate':>10}")
    for quad in sorted(bucket["seat_quad"]):
        wins, played = bucket["seat_quad"][quad]
        print(f"{quad:>10}{played:>10,}{wins:>8,}{wins / played:>10.1%}")
    if bucket["diff_n"]:
        print(f"matches where the two seats DIFFER in quadrant count: "
              f"{bucket['diff_n']:,}")
        print(f"  the seat with MORE quadrants wins: {bucket['diff_more']:,}"
              f"/{bucket['diff_n']:,} = {bucket['diff_more'] / bucket['diff_n']:.1%}")
    print()

    print("--- QUADRANT TIMING (does the purchase PRECEDE the win?) ---")
    for quad, label in ((2, "2nd (NE)"), (3, "3rd (SW)")):
        days = bucket["unlock"].get(quad) or []
        if not days:
            continue
        print(f"  {label:<9} unlocked by {len(days):>5} seats | median day "
              f"{statistics.median(days):>4.0f} (min {min(days):.0f}, max {max(days):.0f})")
    four = bucket["unlock"].get(4) or []
    if four:
        print(f"  {'4th (SE)':<9} bought  by {len(four):>5} seats | median day "
              f"{statistics.median(four):>4.0f} (min {min(four):.0f}, max {max(four):.0f})")
        gaps = [gap for gap in bucket["four_gap"] if gap is not None]
        if gaps:
            ahead = sum(1 for gap in gaps if gap > 0)
            level = sum(1 for gap in gaps if gap == 0)
            print(f"    at the moment of the 4th purchase the buyer was AHEAD: "
                  f"{ahead:,}/{len(gaps):,} = {ahead / len(gaps):.1%} (level: {level:,})")
            print(f"    median bank gap at that moment: ${statistics.median(gaps):+,.0f}"
                  f"   (+ = buyer ahead, - = buyer behind)")
        late = sum(1 for day in four if day > 15)
        print(f"    bought after day 15: {late:,}/{len(four):,} = {late / len(four):.0%}")
    else:
        print("  nobody bought the 4th quadrant in this bucket")
    print()

    if not full:
        return

    print("--- BEHAVIOUR SEPARATORS (winner vs loser WITHIN the same match) ---")
    print(f"{'metric':<20}{'winner':>11}{'loser':>11}{'delta':>11}"
          f"{'winner more':>13}{'wins/losses':>14}")
    rows = []
    for metric in T2_METRICS:
        wvals = bucket["winner_vals"][metric]
        lvals = bucket["loser_vals"][metric]
        wv = statistics.median(wvals) if wvals else 0
        lv = statistics.median(lvals) if lvals else 0
        more = bucket["winner_more"][metric]
        less = bucket["winner_less"][metric]
        share = more / max(1, more + less)
        rows.append((metric, wv, lv, wv - lv, share, more, less))
    for metric, wv, lv, delta, share, more, less in sorted(
            rows, key=lambda row: -abs(row[4] - 0.5)):
        flag = "  <-- separates" if abs(share - 0.5) >= 0.15 else ""
        print(f"{metric:<20}{wv:>11.1f}{lv:>11.1f}{delta:>+11.1f}"
              f"{share:>13.1%}{more:>7}/{less:<7}{flag}")
    print()
    print("'winner more' = share of matches where the WINNER has more of that metric.")
    print("50% = no separation. Only the direction counts: the final ranking is a")
    print("win-rate tournament, not a margin race.")
    print()

    print("--- MARGINS ---")
    print(f"absolute margin, median: ${statistics.median(bucket['margins']):,.0f}")
    print(f"relative margin, median: {statistics.median(bucket['relative']):.2%}"
          f" of the winner's bank")
    close = sum(1 for rel in bucket["relative"] if rel < 0.05)
    print(f"decided by under 5% of the winner's bank: {close:,}/{games:,}"
          f" = {close / games:.0%}")
    print()


# The local 608-episode measurement, taken from the 2026-09-23 dataset (our local
# cache is exactly those files). Which label applies depends on the MOUNTED day, not
# on this constant: a different day is an out-of-sample replication, while the SAME
# day is a same-sample calibration and cannot support any cross-day claim.
LOCAL_0923 = {
    "day": "2026-09-23",
    "episodes": 608,
    "seats": 1216,
    "quad_wins": {2: 0, 3: 334, 4: 274},
    "quad_played": {2: 12, 3: 756, 4: 448},
    "differ_share": 0.672,
    # Measured over the same 608 episodes (scripts/corpus_quadrant_timing.py). The
    # 100-file preview read 16% ahead and a $-3,495 median gap; the full 608 read
    # 21% and $-3,083, so the same-sample value is the one used as the baseline.
    "ahead_at_four": 0.21,
    "gap_at_four": -3083.0,
    "day4_median": 10.0,
    "day4_max": 13.0,
}


def t2_is_same_sample(day):
    # The baseline summarizes ONE published day. A run mounting that exact day is
    # re-measuring the corpus LOCAL_0923 was computed from, so its deltas are
    # calibration noise, not evidence about a different day.
    return day == LOCAL_0923["day"]


def t2_local_table(bucket, local, label):
    # Numbers only, printed next to the local rates so the label can always be
    # checked against the values. Returns the per-quadrant rates and deltas so the
    # caller decides whether a verdict is even meaningful.
    print("--- vs the LOCAL " + local["day"] + " measurement (" + label + ") ---")
    print(f"local: {local['episodes']:,} episodes / {local['seats']:,} seats"
          f"   |   this day: {bucket['games']:,} matches / {2 * bucket['games']:,} seats")
    print(f"{'quadrants':>10}{'local':>12}{'local rate':>12}{'this day':>12}"
          f"{'here':>9}{'delta':>9}")
    here = {}
    deltas = {}
    for quad in (2, 3, 4):
        local_wins = local["quad_wins"].get(quad, 0)
        local_played = local["quad_played"].get(quad, 0)
        local_rate = local_wins / local_played if local_played else 0.0
        wins, played = bucket["seat_quad"].get(quad, [0, 0])
        rate = wins / played if played else None
        here[quad] = rate
        deltas[quad] = None if rate is None else rate - local_rate
        cell = f"{wins}/{played}" if played else "-"
        shown = "n/a" if rate is None else f"{rate:>9.1%}"
        delta = "n/a" if rate is None else f"{rate - local_rate:>+9.1%}"
        print(f"{quad:>10}{local_wins:>5}/{local_played:<6}{local_rate:>12.1%}"
              f"{cell:>12}{shown}{delta}")
    here_differ = bucket["diff_more"] / bucket["diff_n"] if bucket["diff_n"] else None
    shown = "n/a" if here_differ is None else f"{here_differ:>9.1%}"
    delta = ("n/a" if here_differ is None
             else f"{here_differ - local['differ_share']:>+9.1%}")
    print(f"{'differ':>10}{'':>12}{local['differ_share']:>12.1%}"
          f"{(str(bucket['diff_more']) + '/' + str(bucket['diff_n'])):>12}"
          f"{shown}{delta}")

    gaps = [gap for gap in bucket["four_gap"] if gap is not None]
    if gaps:
        ahead = sum(1 for gap in gaps if gap > 0) / len(gaps)
        print(f"4th quadrant, buyer AHEAD at the moment of purchase: local "
              f"{local['ahead_at_four']:.0%} | this day {ahead:.0%}"
              f" ({ahead - local['ahead_at_four']:+.1%})")
        print(f"4th quadrant, median bank gap at that purchase: local "
              f"${local['gap_at_four']:,.0f} | this day "
              f"${statistics.median(gaps):,.0f}")
    four_days = bucket["unlock"].get(4) or []
    if four_days:
        late = sum(1 for day in four_days if day > 15)
        print(f"4th quadrant unlock day: local median {local['day4_median']:.0f}"
              f" (max {local['day4_max']:.0f}; every purchase by day 15) | this day"
              f" median {statistics.median(four_days):.0f}"
              f" (max {max(four_days):.0f}) | after day 15: {late}")

    return here, deltas


def t2_compare(bucket, local):
    # Genuinely different day: a real out-of-sample replication, so the local
    # numbers are an independent comparison and a CONFIRMS / SHRINKS / CONTRADICTS
    # verdict is meaningful.
    here, deltas = t2_local_table(bucket, local, "OUT-OF-SAMPLE")
    if here.get(4) is None or here.get(3) is None:
        verdict = "INDETERMINATE (a quadrant group is empty on this day)"
    elif here[4] <= here[3]:
        verdict = "CONTRADICTS (the 4th quadrant no longer beats the 3rd here)"
    elif any(delta is not None and delta > 0.05 for delta in deltas.values()):
        verdict = "CONFIRMS / STRENGTHENS (direction holds, at least one rate higher)"
    elif all(delta is not None and abs(delta) <= 0.05 for delta in deltas.values()):
        verdict = "CONFIRMS (every quadrant rate within 5 points of the local one)"
    else:
        verdict = "SHRINKS (direction holds, effect smaller than local)"
    print("verdict, this day against the local " + local["day"] + " finding: " + verdict)
    print()


def t2_same_sample(bucket, local):
    # The mounted day IS the corpus the local baseline was measured from. The
    # numbers are a calibration check that this report reproduces the local script;
    # there is deliberately NO verdict, because a day compared against itself
    # confirms and contradicts nothing and implies no cross-day robustness.
    t2_local_table(bucket, local, "SAME-SAMPLE calibration, not a replication")
    print("same-sample calibration: the mounted day is the same corpus the local "
          + local["day"] + " baseline was measured from, so the deltas above only")
    print("check that this report reproduces the local numbers. They are NOT")
    print("independent evidence: a day compared against itself yields no replication")
    print("verdict, and cross-day robustness cannot be inferred from the one day")
    print("mounted here.")
    print()


def t2_per_day(day_buckets):
    # Cheap by construction: the per-day buckets are a few scalars, not replays.
    print("=" * 84)
    print("TIER 2 - PER-DAY QUADRANT WIN RATES (stable across days, or tracking the field?)")
    print("=" * 84)
    print(f"{'day':<12}{'seats':>7}{'2q':>17}{'3q':>17}{'4q':>17}{'differ->more':>14}")
    for day in sorted(day_buckets):
        bucket = day_buckets[day]
        row = f"{day:<12}{2 * bucket['games']:>7,}"
        for quad in (2, 3, 4):
            wins, played = bucket["seat_quad"].get(quad, [0, 0])
            cell = f"{wins}/{played} = {wins / played:.0%}" if played else "-"
            row += f"{cell:>17}"
        differ = (f"{bucket['diff_more'] / bucket['diff_n']:.0%}"
                  if bucket["diff_n"] else "-")
        print(row + f"{differ:>14}")
    print()
    print("The index manifest shows the field drifting (median_avg_score 3,084 on")
    print("09-18 down to 2,983 on 09-24). A rate that holds across every day is a")
    print("robustness statement; a rate that tracks the day's field is a warning that")
    print("the sample is selected.")
    if len(day_buckets) < 2:
        print("Only one day is mounted: cross-day stability cannot be inferred from")
        print("one mounted day, so the row above is a single-day rate, not a")
        print("robustness statement.")
    print()


# Order matters: the newest day leads, alone. Which comparison applies is decided
# by the DAY, not by the code path, so a run can never present the local baseline's
# own day as if it were independent evidence.
if t2_is_same_sample(t2_newest):
    t2_report(t2_day_buckets[t2_newest],
              "TIER 2.1 - DAY " + t2_newest
              + " ALONE (same-sample calibration, NOT out-of-sample)")
    t2_same_sample(t2_day_buckets[t2_newest], LOCAL_0923)
else:
    t2_report(t2_day_buckets[t2_newest],
              "TIER 2.1 - DAY " + t2_newest + " ALONE (out-of-sample replication)")
    t2_compare(t2_day_buckets[t2_newest], LOCAL_0923)

# A pool of one day is that same day counted a second time, so it is skipped
# instead of published as if a wider sample had been measured.
if len(t2_days) > 1:
    t2_report(t2_pool,
              "TIER 2.2 - POOL " + t2_days[0] + " .. " + t2_days[-1]
              + " (INCLUDES the day the local finding came from)")
else:
    print("=" * 84)
    print("TIER 2.2 - POOL " + t2_days[0] + " .. " + t2_days[-1]
          + " (SKIPPED: same single day as TIER 2.1)")
    print("=" * 84)
    print("Only one day is mounted, so the pool would repeat that same day. No")
    print("pooled cross-day number is printed, and the per-day table below cannot")
    print("show cross-day stability.")
    print()
t2_per_day(t2_day_buckets)


### The one-day economic event study

Primary cohort only: exactly one seat ends at four observed quadrants and its
opponent at three, both profiled inside the SAME replay. The tables report cash
(`farm.money`, never revenue), its daily change, the buyer-minus-opponent gap and
observable resources per in-game day and per event offset relative to the buyer's
observed unlock (-4..+8). The purchase day is labelled as containing the
mechanical -$4,000 land cost. Every row prints its own denominator; the skipped
counts and class mix are printed above the tables so a reduced sample cannot pass
for a complete one. This is descriptive: the within-game control removes shared
shops and market, not policy, skill or selection, and repeated teams make rows
non-independent.

In [ ]:
# --- DAILY ECONOMIC EVENT-TIME REPORT (bounded) ---
# Reads the accumulator filled inside the single Tier-2 full-parse pass, so no
# replay is opened twice. Tier-2 skips are added here because a replay that the
# tier-2 profiler could not read was never offered to the economic extractor.
econ_summary["skipped"].update(t2_skipped)
print(render_report(econ_summary, DATA_DATE))


### The one-day shop-window demand study

Filled by the SAME Tier-2 pass, so it costs no extra replay read. Per product and
per seat: the observed shop timeline, expected shop demand from the mounted
engine table, executed units from holdings drops, the shop-window share of those
units, the within-game winner-versus-loser comparison (share and the shop vs
non-shop unit split) and the tape's YARN gate (wool sold on `YARN_STORE` days vs
not). Every row prints its `n`, every skip reason is listed, the demand table's
absence is flagged as a mapping-only fallback, and the report states its own
limits: descriptive, one dataset day, within-game only.

In [ ]:
# --- SHOP-WINDOW DEMAND REPORT (bounded, same single pass) ---
# shop_summary was filled by the SAME Tier-2 loop that filled econ_summary, so
# this costs no second read of a replay. Tier-2 skips are copied in for the same
# reason as the economic report, and the report itself prints nothing causal:
# it labels itself descriptive, one dataset day, within-game only.
#
# shop_render_report, not render_report: both extractor cells share this one
# kernel namespace, so the shop code carries its own prefix. Before that, this
# cell called the SHOP renderer for the ECONOMIC summary and the kernel died on
# KeyError 'demand_table'.
shop_summary["skipped"].update(t2_skipped)
print(shop_render_report(shop_summary, DATA_DATE))


## The published market curve - where marginal value lives

Price is a deterministic function of inventory, and the curve is brutally steep
for shop-drained products while nearly flat for wheat and egg.

In [ ]:
curve = {}
curve_path = (find_all("price_curve.csv") or [None])[0]
if curve_path:
    with open(curve_path) as handle:
        for row in csv.DictReader(handle):
            curve.setdefault(row["product"], {})[int(row["inventory"])] = int(row["price"])
    levels = [9800, 9950, 10000, 10050, 10100, 10500, 12000]
    order = ["STRAWBERRY", "MILK", "WOOL", "MELON", "WHEAT", "EGG", "CARROT", "FERTILIZER"]
    print(f"{'product':<12}" + "".join(f"{lv:>9}" for lv in levels))
    for product in order:
        row = curve.get(product, {})
        print(f"{product:<12}" + "".join(f"{row.get(lv, 0):>9}" for lv in levels))
    print("\nmarginal loss from +100 units of supply at I0:")
    for product in order:
        row = curve.get(product, {})
        base = row.get(10000)
        if base:
            print(f"  {product:<12} {base:>5} -> {row.get(10100, 0):>5}"
                  f"  ({100 * (row.get(10100, 0) - base) / base:+.0f}%)")
else:
    print("price_curve.csv not attached")

## Fertilizer value, priced at the MARGINAL point

The published growth traces give accumulated yield per policy. The extra units must
be valued step by step along the curve, because the last unit of a batch is worth
far less than the first.

In [ ]:
growth_path = (find_all("crop_growth_trace.csv") or [None])[0]
if growth_path:
    by_crop = collections.defaultdict(lambda: collections.defaultdict(list))
    with open(growth_path) as handle:
        for row in csv.DictReader(handle):
            by_crop[row["crop"]][row["policy"]].append(float(row["yield_units"] or 0))

    def price_at(product, inventory):
        row = curve.get(product, {})
        return row.get(inventory, row.get(10000, 0))

    print(f"{'crop':<12}{'no fert':>9}{'fert':>7}{'delta':>7}"
          f"{'marginal +delta (I0->I0+delta)':>34}")
    for crop in sorted(by_crop):
        plain = by_crop[crop].get("watered_no_fertilizer")
        fert = by_crop[crop].get("watered_fertilized")
        if not plain or not fert:
            continue
        delta = int(sum(fert) - sum(plain))
        # THE POINT OF THIS CELL: value the extra units at the curve price AFTER
        # the extra supply lands, not at the pre-supply or base price. Valuing a
        # marginal unit at a flat price is the mistake that produced two wrong
        # conclusions in this project (the first version of this very cell made it).
        if delta > 0:
            entry = price_at(crop, 10000)
            exit_ = price_at(crop, 10000 + delta)
            value = sum(price_at(crop, 10000 + step) for step in range(1, delta + 1))
            print(f"{crop:<12}{sum(plain):>9.0f}{sum(fert):>7.0f}{delta:>+7d}"
                  f"   ${entry:>5} -> ${exit_:<5}  revenue ${value:>7,.0f}"
                  f"  vs ~$42 forgone")
        else:
            print(f"{crop:<12}{sum(plain):>9.0f}{sum(fert):>7.0f}{delta:>+7d}"
                  f"   no gain")
else:
    print("crop_growth_trace.csv not attached")

## How large a fertilizer programme can the market absorb?

Extra production only pays while the marginal unit still beats the fertilizer it
costs. The market's absorption comes from the shops (each type present with
probability 65.6%) plus the town centre, and the centre's real rate is one unit per
product per 24 turns - flat - not the x2/x4 schedule the stale CSV claims.

In [ ]:
# How big can a fertilizer programme be before the last unit stops paying?
#
# Assumptions, stated because this is a first-order model: the market sits around
# I0 (our own sale-time inventories were 9,980-10,075, which supports it), we are
# the marginal supplier, and the opponent's supply is unchanged. The benchmark is
# what finally decides; this only sizes the programme.
demand_path = (find_all("town_shop_demand.csv") or [None])[0]
FERT_COST = 42.0        # what we forgo by not selling that fertilizer unit
growth_path = (find_all("crop_growth_trace.csv") or [None])[0]

if demand_path and growth_path and curve:
    P_UNLOCK = 1 - (7 / 8) ** 8     # 8 draws over 8 shop types, with replacement
    SHOP_DAYS = 27                  # shops unlock on days 3, 6, ..., 24
    # The engine's town centre takes ONE unit of each product per 24 turns, flat,
    # all season. town_center_demand_schedule.csv claims interval 12 with x2/x4
    # day multipliers - that table is STALE and would overstate the drain 2-8x.
    CENTRE_SEASON = 1.0 * 30

    shop_rate = collections.Counter()
    with open(demand_path) as handle:
        for row in csv.DictReader(handle):
            if row["source"] == "SHOP":
                shop_rate[row["product"]] += float(row["units_per_day_if_unlocked"]) * P_UNLOCK

    by_crop = collections.defaultdict(lambda: collections.defaultdict(list))
    with open(growth_path) as handle:
        for row in csv.DictReader(handle):
            by_crop[row["crop"]][row["policy"]].append(float(row["yield_units"] or 0))

    print(f"{'crop':<12}{'drain/szn':>10}{'per appl':>9}{'profitable u':>13}"
          f"{'appl':>6}{'revenue':>10}{'cost':>8}{'net':>9}")
    for crop in sorted(by_crop):
        row = curve.get(crop)
        plain = by_crop[crop].get("watered_no_fertilizer")
        fert = by_crop[crop].get("watered_fertilized")
        if not row or not plain or not fert:
            continue
        per_app = int(sum(fert) - sum(plain))
        if per_app <= 0:
            continue
        # The k-th extra unit meets the inventory it helps create, so it is worth
        # curve(I0 + k). Walk outward until a unit no longer beats the fertilizer.
        revenue = 0.0
        units = 0
        for step in range(1, 600):
            price = row.get(10000 + step)
            if price is None or price < FERT_COST:
                break
            revenue += price
            units = step
        apps = units // per_app
        drain = shop_rate[crop] * SHOP_DAYS + CENTRE_SEASON
        print(f"{crop:<12}{drain:>10.0f}{per_app:>9d}{units:>13d}{apps:>6d}"
              f"{revenue:>10,.0f}{apps * FERT_COST:>8,.0f}"
              f"{revenue - apps * FERT_COST:>9,.0f}")
    print()
    print("'profitable u' = extra units whose marginal price still beats the")
    print("fertilizer it costs; beyond that the curve has collapsed and the last")
    print("unit decides the profit, not the average.")
else:
    print("town_shop_demand.csv, crop_growth_trace.csv or price_curve.csv not attached")

## The measured reference league (a diverse opponent ladder)

In [ ]:
for name in ("crop_economics.csv", "head_to_head_games.csv",
             "baseline_league.csv", "agents_manifest.csv"):
    path = (find_all(name) or [None])[0]
    if not path:
        print(f"{name:<26} not attached")
        continue
    with open(path) as handle:
        rows = list(csv.DictReader(handle))
    print(f"{name:<26} {len(rows):>5} rows  columns: {list(rows[0].keys())[:8]}")

league_path = (find_all("baseline_league.csv") or [None])[0]
if league_path:
    with open(league_path) as handle:
        league = list(csv.DictReader(handle))
    ranked = collections.Counter()
    played = collections.Counter()
    for row in league:
        try:
            games = int(row["games"] or 0)
        except ValueError:
            continue
        # Count BOTH sides: an agent mostly sits in column b, and reading only the
        # agent_a columns printed five of the ten agents as 0 wins out of 50-odd.
        for wins_key, name_key in (("wins_a", "agent_a"), ("wins_b", "agent_b")):
            try:
                ranked[row[name_key]] += int(row[wins_key] or 0)
                played[row[name_key]] += games
            except ValueError:
                continue
    print("\nreference league win rate (measured, this is the opponent ladder):")
    table = sorted(
        ((100 * wins / max(played[agent], 1), agent, wins, max(played[agent], 1))
         for agent, wins in ranked.items()),
        reverse=True,
    )
    for rate, agent, wins, total in table:
        print(f"  {agent:<20} {wins:>5}W / {total:>5}  = {rate:>5.1f}%")

## Next steps

1. Raise `MAX_EPISODES` once the tier-1 cost is known (the cell prints the rate and
   projects the whole league), and attach more daily datasets to widen the window.
2. Tier 2 now runs the within-game behaviour analysis (quadrant dose-response,
   unlock timing, separators, margins) over every mounted day, leading with the
   newest day: same-sample calibration when it is the local baseline's own date,
   out-of-sample replication otherwise. Raise `MAX_REPLAYS_T2` to widen the sample,
   and keep it above the mounted replay count so the bound never bites by accident.
3. Upload our own replay caches as a private dataset so our games live here too.
4. The GPU's only honest use here is a market surrogate trained on trajectories;
   everything above is CPU work.